# Deep Learning

---
## Setup, Data, and Frozen CLIP
---

This notebook integrates the baseline arithmetic retrieval, SLERP variants, the multiple-SLERP grid, and the neural prompt-search finetuning experiment into one executable pipeline.

### Libraries

In [1]:
import json
import random
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Optional, Sequence

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.tensorboard import SummaryWriter

import matplotlib.pyplot as plt
from torchvision.datasets import CelebA
import transformers
from transformers import CLIPModel, CLIPProcessor
import torchvision.transforms as transforms
from torchvision.transforms import InterpolationMode

%matplotlib inline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Use the GPU whenever it is available. The evaluation code below is deliberately batched
# so a 14/15 GB card can spend most of its time on large matrix multiplications.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"torch={torch.__version__}, transformers={transformers.__version__}")

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")
    gpu = torch.cuda.get_device_properties(0)
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {gpu.total_memory / 1024**3:.1f} GB")

Using device: cuda
torch=2.12.1+cu126, transformers=5.12.1
CUDA device: Tesla T4
CUDA memory: 15.6 GB


### Data Paths

In [2]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)


def first_existing(paths: Sequence[Path], fallback: Path) -> Path:
    """Pick the first existing path, while keeping a deterministic fallback for clear errors."""
    for candidate in paths:
        if candidate.exists():
            return candidate
    return fallback


if IN_COLAB:
    # Colab keeps large archives and the JSON on Drive, then extracts into local runtime storage.
    DATA_ROOT = Path("/content/datasets")
    DRIVE_DATA_ROOT = Path("/content/drive/MyDrive/deep_learning/data")

    CELEBA_ZIP = DRIVE_DATA_ROOT / "celeba.zip"
    FROZEN_DATA_ZIP = DRIVE_DATA_ROOT / "frozen_data.zip"
    ANNOTATIONS_PATH = DRIVE_DATA_ROOT / "celeba_evaluation.json"

    CELEBA_ROOT = DATA_ROOT
    FROZEN_DATA_ROOT = DATA_ROOT / "frozen_data"
else:
    # The lab VM path is the expected layout, with local fallbacks for running the notebook elsewhere.
    local_candidates = [
        Path("/home/disi/deep_learning/DeepLearning2026/data"),
        Path.cwd() / "data",
        Path.cwd().parent / "data",
    ]
    DATA_ROOT = first_existing(local_candidates, local_candidates[0])

    CELEBA_ZIP = DATA_ROOT / "celeba.zip"
    FROZEN_DATA_ZIP = DATA_ROOT / "frozen_data.zip"
    ANNOTATIONS_PATH = DATA_ROOT / "celeba_evaluation.json"

    CELEBA_ROOT = DATA_ROOT / "celeba"
    FROZEN_DATA_ROOT = DATA_ROOT / "frozen_data"

if IN_COLAB:
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    FROZEN_DATA_ROOT.mkdir(parents=True, exist_ok=True)


def describe_path(label: str, path: Path) -> None:
    print(f"{label}: {path} | exists={path.exists()} | is_dir={path.is_dir()}")


print("Path mode:", "Colab/Drive" if IN_COLAB else "VM/local")
describe_path("DATA_ROOT", DATA_ROOT)
describe_path("CELEBA_ROOT", CELEBA_ROOT)
print(f"ANNOTATIONS_PATH: {ANNOTATIONS_PATH} | exists={ANNOTATIONS_PATH.exists()} | is_file={ANNOTATIONS_PATH.is_file()}")
describe_path("FROZEN_DATA_ROOT", FROZEN_DATA_ROOT)
print(f"CELEBA_ZIP: {CELEBA_ZIP} | exists={CELEBA_ZIP.exists()}")
print(f"FROZEN_DATA_ZIP: {FROZEN_DATA_ZIP} | exists={FROZEN_DATA_ZIP.exists()}")

Path mode: VM/local
DATA_ROOT: /home/disi/deep_learning/DeepLearning2026/data | exists=True | is_dir=True
CELEBA_ROOT: /home/disi/deep_learning/DeepLearning2026/data/celeba | exists=True | is_dir=True
ANNOTATIONS_PATH: /home/disi/deep_learning/DeepLearning2026/data/celeba_evaluation.json | exists=True | is_file=True
FROZEN_DATA_ROOT: /home/disi/deep_learning/DeepLearning2026/data/frozen_data | exists=True | is_dir=True
CELEBA_ZIP: /home/disi/deep_learning/DeepLearning2026/data/celeba.zip | exists=True
FROZEN_DATA_ZIP: /home/disi/deep_learning/DeepLearning2026/data/frozen_data.zip | exists=True


In [3]:
def unzip_if_needed(zip_path: Path, target_dir: Path, marker_glob: str, description: str) -> None:
    """Extract a zip file only when the expected files are not already present."""
    target_dir.mkdir(parents=True, exist_ok=True)
    if any(target_dir.rglob(marker_glob)):
        print(f"{description}: already present in {target_dir}")
        return
    if not zip_path.exists():
        print(f"{description}: zip not found at {zip_path}. Skipping extraction.")
        return
    print(f"{description}: extracting {zip_path.name} -> {target_dir}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(target_dir)


if IN_COLAB:
    unzip_if_needed(CELEBA_ZIP, CELEBA_ROOT, "*.jpg", "CelebA images")
    unzip_if_needed(FROZEN_DATA_ZIP, FROZEN_DATA_ROOT, "*.pt", "Frozen CLIP visual features")
else:
    print("VM/local mode: using pre-extracted data; no zip extraction will be attempted.")

VM/local mode: using pre-extracted data; no zip extraction will be attempted.


In [4]:
def find_pt_directory(root: Path) -> Path:
    """Find the directory that actually contains the frozen CLIP .pt image embeddings."""
    if any(root.glob("*.pt")):
        return root
    candidates = [p.parent for p in root.rglob("*.pt")]
    if not candidates:
        raise FileNotFoundError(
            f"No .pt visual feature files found under {root}. "
            "Check FROZEN_DATA_ROOT or extract frozen_data.zip."
        )
    return max(set(candidates), key=lambda p: sum(1 for _ in p.glob("*.pt")))


VISUAL_FEATURE_DIR = find_pt_directory(FROZEN_DATA_ROOT)
print("Using visual features from:", VISUAL_FEATURE_DIR)
print("Number of .pt files:", sum(1 for _ in VISUAL_FEATURE_DIR.glob("*.pt")))

if not ANNOTATIONS_PATH.exists():
    raise FileNotFoundError(f"Missing annotations file: {ANNOTATIONS_PATH}")

with open(ANNOTATIONS_PATH, "r") as f:
    annotations = json.load(f)

print(f"Loaded {len(annotations)} evaluation queries")
queries_df = pd.DataFrame({"query_id": range(len(annotations)), "query": [a["query"] for a in annotations]})
queries_df

Using visual features from: /home/disi/deep_learning/DeepLearning2026/data/frozen_data
Number of .pt files: 19962
Loaded 14 evaluation queries


,query_id,query
0,0,+Smiling
1,1,+Eyeglasses
2,2,-Heavy_Makeup
3,3,+Male
4,4,-Young
5,5,+Blond_Hair
6,6,+Mustache
7,7,-Young
8,8,"+Eyeglasses, +Smiling"
9,9,"+Black_Hair, -Wavy_Hair"


### Frozen CLIP

In [5]:
MODEL_NAME = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(MODEL_NAME).to(device).eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor

# The CLIP backbone is frozen for every experiment. The finetuning section only trains
# continuous prompt tokens and small residual modules around the frozen embedding space.
for p in model.parameters():
    p.requires_grad_(False)

TEXT_WIDTH = model.config.text_config.hidden_size
PROJECTION_DIM = model.config.projection_dim
MAX_TEXT_LEN = model.config.text_config.max_position_embeddings
EOT_TOKEN_ID = model.config.text_config.eos_token_id

print("Loaded", MODEL_NAME)
print("text hidden width:", TEXT_WIDTH)
print("projection dim:", PROJECTION_DIM)
print("max text length:", MAX_TEXT_LEN)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loaded openai/clip-vit-base-patch32
text hidden width: 512
projection dim: 512
max text length: 77


### Tensor Utilities and Image Bank

Frozen image embeddings are loaded once, normalized once, and then reused by every retrieval method.

In [6]:
def safe_torch_load(path: Path, map_location="cpu"):
    """torch.load wrapper compatible with old and new PyTorch versions."""
    try:
        # weights_only=True avoids unpickling arbitrary Python objects in newer PyTorch versions.
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        # Older PyTorch versions do not have weights_only, so fall back to the older signature.
        return torch.load(path, map_location=map_location)


def normalize(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    # clamp_min keeps zero or almost-zero vectors from producing NaNs during division.
    return x / x.norm(dim=-1, keepdim=True).clamp_min(eps)


def as_feature_tensor(model_output, clip_model: nn.Module) -> torch.Tensor:
    """Extract projected CLIP text features from tensor or Hugging Face output objects."""
    if isinstance(model_output, torch.Tensor):
        return model_output
    if hasattr(model_output, "text_embeds") and model_output.text_embeds is not None:
        return model_output.text_embeds
    if hasattr(model_output, "pooler_output") and model_output.pooler_output is not None:
        if not hasattr(clip_model, "text_projection"):
            raise TypeError("CLIP text pooler output needs model.text_projection, but it is missing")
        return clip_model.text_projection(model_output.pooler_output)
    if isinstance(model_output, dict):
        for key in ("text_embeds", "pooler_output"):
            if key in model_output and model_output[key] is not None:
                if key == "pooler_output":
                    if not hasattr(clip_model, "text_projection"):
                        raise TypeError("CLIP text pooler output needs model.text_projection, but it is missing")
                    return clip_model.text_projection(model_output[key])
                return model_output[key]
    if isinstance(model_output, (tuple, list)) and model_output:
        return model_output[0]
    raise TypeError(f"Could not extract a tensor from model output type {type(model_output)!r}")


@torch.no_grad()
def encode_text(prompts: list[str] | str, batch_size: int = 64) -> torch.Tensor:
    """Encode prompt strings with frozen CLIP and L2-normalize the result."""
    single = isinstance(prompts, str)
    if single:
        prompts = [prompts]

    feats = []
    for start in range(0, len(prompts), batch_size):
        # Text encoding is batched for memory stability when we cache many templates.
        batch = prompts[start:start + batch_size]
        inputs = processor(text=batch, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        text_features = as_feature_tensor(model.get_text_features(**inputs), model)
        feats.append(normalize(text_features.float()).detach())

    out = torch.cat(feats, dim=0)
    return out[0:1] if single else out


def slerp_vectors(v0: torch.Tensor, v1: torch.Tensor, alpha: float, eps: float = 1e-6) -> torch.Tensor:
    """
    Numerically stable spherical linear interpolation.

    alpha=0.0 returns v0; alpha=1.0 returns v1. Inputs can be [D] or [N, D].
    """
    if not 0.0 <= float(alpha) <= 1.0:
        raise ValueError(f"alpha must be in [0, 1], got {alpha}")

    squeeze = False
    if v0.ndim == 1:
        v0 = v0.unsqueeze(0)
        squeeze = True
    if v1.ndim == 1:
        v1 = v1.unsqueeze(0)

    v0 = normalize(v0)
    v1 = normalize(v1)
    dot = (v0 * v1).sum(dim=-1, keepdim=True).clamp(-1.0, 1.0)

    # When vectors are almost parallel, linear interpolation is more stable than dividing
    # by a tiny sin(theta). torch.where keeps the batch path vectorized.
    close = dot.abs() > 0.9995
    theta = torch.acos(dot)
    sin_theta = torch.sin(theta).clamp_min(eps)

    s0 = torch.sin((1.0 - alpha) * theta) / sin_theta
    s1 = torch.sin(alpha * theta) / sin_theta
    spherical = s0 * v0 + s1 * v1
    linear = (1.0 - alpha) * v0 + alpha * v1

    out = normalize(torch.where(close, linear, spherical))
    return out.squeeze(0) if squeeze else out


# Alias used by the finetuning section imported from the dedicated SLERP notebook.
slerp = slerp_vectors


class ImageEmbeddings:
    """In-memory store of frozen CLIP image embeddings with fast ID-to-row lookup."""

    def __init__(self, ids: list[int], features: torch.Tensor):
        self.ids = [int(i) for i in ids]
        self.embeddings = normalize(features.to(device))
        self.id_to_row = {int(idx): row for row, idx in enumerate(self.ids)}
        self.id_tensor = torch.tensor(self.ids, dtype=torch.long, device=device)
        self._row_lookup = None
        if self.ids and min(self.ids) >= 0:
            max_id = max(self.ids)
            # CelebA IDs are dense enough that this small lookup table removes Python work
            # from batched reference lookup and source-image exclusion.
            if max_id <= max(1_000_000, 4 * len(self.ids)):
                lookup = torch.full((max_id + 1,), -1, dtype=torch.long, device=device)
                lookup[self.id_tensor] = torch.arange(len(self.ids), dtype=torch.long, device=device)
                self._row_lookup = lookup

    @classmethod
    def from_directory(cls, directory: Path, max_images: Optional[int] = None):
        pt_files = sorted(directory.glob("*.pt"), key=lambda p: int(p.stem))
        if max_images is not None:
            pt_files = pt_files[:max_images]
        if not pt_files:
            raise FileNotFoundError(f"No .pt files found in {directory}")

        ids, feats = [], []
        for pt_file in tqdm(pt_files, desc="Loading frozen visual features"):
            tensor = safe_torch_load(pt_file, map_location="cpu")
            if isinstance(tensor, dict):
                tensor = tensor.get("features", tensor.get("image_features"))
            if tensor is None:
                raise ValueError(f"Could not read tensor from {pt_file}")
            ids.append(int(pt_file.stem))
            feats.append(tensor.float().view(-1))
        return cls(ids, torch.stack(feats, dim=0))

    def rows_for_ids(self, image_ids: list[int | str] | torch.Tensor) -> torch.Tensor:
        if torch.is_tensor(image_ids):
            ids_t = image_ids.to(device=device, dtype=torch.long)
        else:
            ids_t = torch.tensor([int(image_id) for image_id in image_ids], dtype=torch.long, device=device)
        if self._row_lookup is not None and ids_t.numel() and int(ids_t.min()) >= 0 and int(ids_t.max()) < self._row_lookup.numel():
            rows = self._row_lookup[ids_t]
            if bool((rows >= 0).all()):
                return rows
        return torch.tensor([self.id_to_row[int(image_id)] for image_id in ids_t.detach().cpu().tolist()], dtype=torch.long, device=device)

    def get(self, image_id: int | str) -> torch.Tensor:
        row = self.id_to_row[int(image_id)]
        return self.embeddings[row:row + 1]

    def batch_get(self, image_ids: list[int | str] | torch.Tensor) -> torch.Tensor:
        return self.embeddings[self.rows_for_ids(image_ids)]

    def topk(self, query_vec: torch.Tensor, k: int, exclude: Optional[set[int]] = None) -> dict[int, float]:
        query_vec = normalize(query_vec.to(device))
        scores = (query_vec @ self.embeddings.T).squeeze(0)
        if exclude:
            scores = scores.clone()
            rows = self.rows_for_ids(list(exclude))
            rows = rows[(rows >= 0) & (rows < scores.numel())]
            scores[rows] = -float("inf")
        values, rows = torch.topk(scores, k=min(k, scores.numel()))
        return {self.ids[int(row)]: float(val) for row, val in zip(rows.detach().cpu(), values.detach().cpu())}


image_embeddings = ImageEmbeddings.from_directory(VISUAL_FEATURE_DIR)
print(f"Image embeddings: {len(image_embeddings.ids)} images, dim={image_embeddings.embeddings.shape[1]}")

Loading frozen visual features:   0%|          | 0/19962 [00:00<?, ?it/s]

Image embeddings: 19962 images, dim=512


### Signed Queries, Cases, and Prompt Text

Queries arrive as comma-separated signed attributes such as `+Smiling,-Eyeglasses`. The case builders below convert the JSON benchmark into a flat list of reference/query/ground-truth cases used by every method.

In [7]:
@dataclass(frozen=True)
class SignedAttribute:
    sign: str
    raw: str


def parse_query(query: str) -> list[SignedAttribute]:
    """Parse '+Smiling,-Eyeglasses' into signed attributes."""
    attrs = []
    for part in query.split(","):
        part = part.strip()
        if not part:
            continue
        if part[0] not in {"+", "-"}:
            raise ValueError(f"Query component must start with + or -: {part!r}")
        attrs.append(SignedAttribute(part[0], part[1:].strip()))
    return attrs


def get_unsigned_queries(annotations: list[dict]) -> list[str]:
    # Preserve first-seen order so text-bank rows and printed debug output are stable.
    seen = []
    for item in annotations:
        for attr in parse_query(item["query"]):
            if attr.raw not in seen:
                seen.append(attr.raw)
    return seen


def attr_phrase(raw_attr: str) -> str:
    return raw_attr.replace("_", " ").lower()


def positive_suffix(raw_attr: str) -> str:
    """Natural positive attribute suffix used by the CoOp-style prompt learner."""
    attr = attr_phrase(raw_attr)
    if attr == "smiling":
        return "a face that is smiling."
    if attr == "male":
        return "a male face."
    if attr == "young":
        return "a young face."
    if attr.startswith("wearing "):
        return f"a face {attr}."
    return f"a face with {attr}."


def negative_suffix(raw_attr: str) -> str:
    """Natural negative/opposite attribute suffix used by the CoOp-style prompt learner."""
    attr = attr_phrase(raw_attr)
    if attr == "male":
        return "a female face."
    if attr == "young":
        return "an old face."
    if attr == "smiling":
        return "a serious unsmiling face."
    return f"a face without {attr}."


def positive_prompt(raw_attr: str) -> str:
    # Remove only the final full stop because CLIP prompt features are cached as plain phrases.
    return positive_suffix(raw_attr).rstrip(".")


def negative_prompt_templates(raw_attr: str) -> list[str]:
    """Fixed-size set of candidate negative prompts for a signed negative attribute."""
    attr = attr_phrase(raw_attr)
    return [
        negative_suffix(raw_attr).rstrip("."),
        f"a face without {attr}",
        f"a face that is not {attr}",
        f"an image without {attr}",
        f"a photo of a person without {attr}",
        f"not {attr}",
    ]


def composed_query_prompt(query: str) -> str:
    """Human-readable whole-query prompt, useful for debugging and optional experiments."""
    chunks = []
    for attr in parse_query(query):
        if attr.sign == "+":
            chunks.append(positive_prompt(attr.raw).replace("a face ", ""))
        else:
            chunks.append(negative_prompt_templates(attr.raw)[0].replace("a face ", ""))
    return "a face " + " and ".join(chunks)


@dataclass(frozen=True)
class RetrievalCase:
    query_id: int
    query: str
    reference_id: int
    ground_truth: list[int]


def make_cases(
    annotations: list[dict],
    image_embeddings: ImageEmbeddings,
    query_indices: Optional[list[int]] = None,
    max_references_per_query: Optional[int] = None,
) -> list[RetrievalCase]:
    """Flatten the JSON benchmark into retrieval cases, skipping IDs missing from the image bank."""
    if query_indices is None:
        query_indices = list(range(len(annotations)))

    cases = []
    for query_id in query_indices:
        item = annotations[int(query_id)]
        ref_ids = list(item["ground_truth"].keys())
        if max_references_per_query is not None:
            ref_ids = ref_ids[:max_references_per_query]
        for ref_id in ref_ids:
            if int(ref_id) not in image_embeddings.id_to_row:
                continue
            gt_ids = [int(x) for x in item["ground_truth"][ref_id] if int(x) in image_embeddings.id_to_row]
            if gt_ids:
                cases.append(RetrievalCase(int(query_id), item["query"], int(ref_id), gt_ids))
    return cases


def make_case(query_id: int, reference_id: int | str) -> RetrievalCase:
    """Build one retrieval case for qualitative inspection."""
    item = annotations[int(query_id)]
    reference_key = str(reference_id)
    if reference_key not in item["ground_truth"]:
        raise KeyError(f"Reference {reference_id} is not part of query {query_id}")
    gt_ids = [int(x) for x in item["ground_truth"][reference_key] if int(x) in image_embeddings.id_to_row]
    if int(reference_id) not in image_embeddings.id_to_row:
        raise KeyError(f"Reference {reference_id} is missing from frozen image embeddings")
    if not gt_ids:
        raise ValueError(f"Reference {reference_id} for query {query_id} has no ground-truth IDs in the image bank")
    return RetrievalCase(int(query_id), item["query"], int(reference_id), gt_ids)


def reference_options(query_id: int) -> list[int]:
    """List valid reference image IDs for a query, useful when choosing qualitative examples."""
    return [int(ref_id) for ref_id in annotations[int(query_id)]["ground_truth"].keys() if int(ref_id) in image_embeddings.id_to_row]


def split_cases(cases: list[RetrievalCase], validation_fraction: float = 0.2, seed: int = SEED):
    """Shuffle once and split cases for prompt-search training/validation."""
    cases = list(cases)
    rng = random.Random(seed)
    rng.shuffle(cases)
    split_at = max(1, int(round(len(cases) * (1.0 - validation_fraction))))
    return cases[:split_at], cases[split_at:]


def split_cases_by_query(
    cases: list[RetrievalCase],
    train_fraction: float = 0.70,
    validation_fraction: float = 0.15,
    seed: int = SEED,
) -> tuple[list[RetrievalCase], list[RetrievalCase], list[RetrievalCase]]:
    """
    Stratified train/validation/test split over reference cases inside each query.

    This is the default split for finetuning: each query keeps held-out references for
    validation and test when possible, so the per-query tables remain meaningful.
    """
    if not 0.0 < train_fraction < 1.0:
        raise ValueError("train_fraction must be in (0, 1)")
    if not 0.0 <= validation_fraction < 1.0:
        raise ValueError("validation_fraction must be in [0, 1)")
    if train_fraction + validation_fraction >= 1.0:
        raise ValueError("train_fraction + validation_fraction must leave room for a test split")

    rng = random.Random(seed)
    buckets: dict[int, list[RetrievalCase]] = {}
    for case in cases:
        buckets.setdefault(case.query_id, []).append(case)

    train_cases, validation_cases, test_cases = [], [], []
    for query_id in sorted(buckets):
        query_cases = list(buckets[query_id])
        rng.shuffle(query_cases)
        n = len(query_cases)

        if n == 1:
            # With a single case there is no honest held-out split; keep it train-only.
            train_cases.extend(query_cases)
            continue
        if n == 2:
            # Keep at least one held-out case if a query is extremely small.
            train_cases.append(query_cases[0])
            test_cases.append(query_cases[1])
            continue

        n_val = max(1, int(round(n * validation_fraction)))
        n_test = max(1, int(round(n * (1.0 - train_fraction - validation_fraction))))

        # Guarantee at least one training case for every query while preserving held-out cases.
        while n - n_val - n_test < 1:
            if n_val >= n_test and n_val > 1:
                n_val -= 1
            elif n_test > 1:
                n_test -= 1
            else:
                break
        n_train = n - n_val - n_test

        train_cases.extend(query_cases[:n_train])
        validation_cases.extend(query_cases[n_train:n_train + n_val])
        test_cases.extend(query_cases[n_train + n_val:])

    rng.shuffle(train_cases)
    rng.shuffle(validation_cases)
    rng.shuffle(test_cases)
    return train_cases, validation_cases, test_cases


raw_attributes = get_unsigned_queries(annotations)
print(f"Attributes in evaluation queries: {len(raw_attributes)}")
raw_attributes

Attributes in evaluation queries: 12


['Smiling',
 'Eyeglasses',
 'Heavy_Makeup',
 'Male',
 'Young',
 'Blond_Hair',
 'Mustache',
 'Black_Hair',
 'Wavy_Hair',
 'Chubby',
 'Wearing_Hat',
 'Wearing_Lipstick']

### Text Feature Bank

In [8]:
def build_text_bank(raw_attributes: list[str]) -> dict:
    """Precompute all frozen CLIP text features needed by the non-learned methods."""
    prompts = []
    keys = []

    for raw in raw_attributes:
        # Label-only feature: closest to the original baseline arithmetic style.
        keys.append(("label", raw, None))
        prompts.append(attr_phrase(raw))

        # Positive prompt feature used by text-SLERP methods.
        keys.append(("positive", raw, None))
        prompts.append(positive_prompt(raw))

        for template_idx, prompt in enumerate(negative_prompt_templates(raw)):
            keys.append(("negative_template", raw, template_idx))
            prompts.append(prompt)

    encoded = encode_text(prompts)

    bank = {
        "raw_attributes": list(raw_attributes),
        "label": {},
        "positive": {},
        "negative_templates": {},
        "negative_basic": {},
        "negative_template_text": {},
    }

    for key, feat in zip(keys, encoded):
        kind, raw, template_idx = key
        feat = feat.unsqueeze(0)
        if kind == "label":
            bank["label"][raw] = feat
        elif kind == "positive":
            bank["positive"][raw] = feat
        else:
            bank["negative_templates"].setdefault(raw, []).append(feat.squeeze(0))
            bank["negative_template_text"].setdefault(raw, []).append(negative_prompt_templates(raw)[template_idx])

    for raw in raw_attributes:
        templates = normalize(torch.stack(bank["negative_templates"][raw], dim=0).to(device))
        bank["negative_templates"][raw] = templates
        # Basic negative = uniform template ensemble before any learned prompt tuning.
        bank["negative_basic"][raw] = normalize(templates.mean(dim=0, keepdim=True))

    return bank


text_bank = build_text_bank(raw_attributes)
print(f"Built text bank for {len(raw_attributes)} attributes")
print(raw_attributes)

Built text bank for 12 attributes
['Smiling', 'Eyeglasses', 'Heavy_Makeup', 'Male', 'Young', 'Blond_Hair', 'Mustache', 'Black_Hair', 'Wavy_Hair', 'Chubby', 'Wearing_Hat', 'Wearing_Lipstick']


---
## Metric
---

The retrieval metric is copied from the baseline notebook and intentionally left unchanged.

In [9]:
def evaluate_retrieval(
    retrieved_indices: list[int],
    ground_truth_indices: list[int],
    k: int
):
    """
    Evaluate the retrieval performance for a single source image.

    Args:
    ----
        retrieved_indices: list of image IDs predicted by the model,
            ordered by similarity (descending).
        ground_truth_indices: list of valid target IDs from the benchmark JSON.
        k: the cutoff for top-K evaluation (e.g., 1, 5, 10).

    Return:
    ------
        A dictionary containing Recall@K and Precision@K.

    """
    # Isolate the top K predictions
    top_k_retrieved = retrieved_indices[:k]

    # Calculate the intersection between predictions and ground truth
    hits = set(top_k_retrieved).intersection(set(ground_truth_indices))
    num_hits = len(hits)

    # Metrics calculations
    # Recall@K (Hit Rate): 1 if at least one match is found, 0 otherwise
    recall_at_k = 1 if num_hits > 0 else 0

    # Precision@K: Fraction of top K predictions that are correct
    precision_at_k = num_hits / k

    return {
        f"Recall@{k}": recall_at_k,
        f"Precision@{k}": precision_at_k
    }

# --- Example Usage ---
# Suppose the model returns these indices from most to least similar:
# predictions = [1, 2, 3, 4, 5]
# And we load this from our JSON for this specific source:
# ground_truth = [3, 2, 1]

# Evaluate at K=1 and K=5
# print("Results @ 1:", evaluate_retrieval(predictions, ground_truth, k=1))
# print("Results @ 5:", evaluate_retrieval(predictions, ground_truth, k=5))

---
## Batched Quantitative and Qualitative Evaluation
---

Every quantitative section below uses these same helpers: build all target vectors for a method in batches, multiply once against the full image bank, take top-k, then score with the unchanged metric.

In [10]:
KS = (1, 5, 10)
# 512 is a good default for a 14/15 GB GPU with CLIP ViT-B/32 embeddings.
# Lower it if another process is using VRAM; raise it if the card has more room.
EVAL_BATCH_SIZE = 512
EXCLUDE_REFERENCE = True


@dataclass(frozen=True)
class MethodSpec:
    # One row in the experiment grid. random methods skip target construction and sample rankings.
    name: str
    prediction_mode: str = "composition"   # "composition" or "random"
    text_mode: str = "arithmetic"          # "arithmetic" or "slerp" or "img2text"
    fusion_mode: str = "add"               # "add" or "slerp"
    negative_weight: float = 0.5            # only used by text_mode="slerp"
    image_alpha: float = 0.5                # only used by fusion_mode="slerp"; alpha toward image
    image_scale: float = 1.0                # optional ablation knob for arithmetic fusion


def method_config_row(method: MethodSpec) -> dict:
    """Return readable method settings for registry and result dataframes."""
    return {
        "method": method.name,
        "prediction_mode": method.prediction_mode,
        "text_mode": method.text_mode if method.prediction_mode != "random" else None,
        "fusion_mode": method.fusion_mode if method.prediction_mode != "random" else None,
        "negative_weight": method.negative_weight if method.text_mode == "slerp" and method.prediction_mode != "random" else None,
        "image_alpha": method.image_alpha if method.fusion_mode == "slerp" and method.prediction_mode != "random" else None,
        "image_scale": method.image_scale if method.fusion_mode == "add" and method.prediction_mode != "random" else None,
    }


def spherical_weighted_average(vectors: list[torch.Tensor], weights: list[float]) -> torch.Tensor:
    """Incremental weighted average on the hypersphere using SLERP."""
    if len(vectors) == 0:
        raise ValueError("Need at least one vector")
    if len(vectors) == 1:
        return normalize(vectors[0])

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device).clamp_min(1e-8)
    weights_t = weights_t / weights_t.sum()

    current = normalize(vectors[0])
    cumulative = weights_t[0]
    for vec, w in zip(vectors[1:], weights_t[1:]):
        # The new vector receives its share of the cumulative mass seen so far.
        alpha = float(w / (cumulative + w))
        current = slerp_vectors(current, vec, alpha=alpha)
        cumulative = cumulative + w
    return normalize(current)


def text_direction_arithmetic(query: str, bank: dict) -> torch.Tensor:
    """Baseline signed text arithmetic: +attribute vectors and -attribute vectors."""
    direction = torch.zeros_like(next(iter(bank["label"].values()))).to(device)
    for attr in parse_query(query):
        if attr.sign == "+":
            direction = direction + bank["label"][attr.raw].to(device)
        else:
            direction = direction - bank["label"][attr.raw].to(device)
    return normalize(direction)
    
def query_direction_img2text(query_index, direction):
    """Turn a signed query into one CLIP text-direction vector on CPU."""
    if query_index in QUERY_DIRECTION_CACHE:
        return QUERY_DIRECTION_CACHE[query_index]

    for sign, attribute in parse_signed_query(annotations[query_index]["query"]):
        direction = direction + sign * text_embedding(attribute)

    QUERY_DIRECTION_CACHE[query_index] = direction
    return direction

def text_direction_slerp(query: str, bank: dict, negative_weight: float = 0.5) -> torch.Tensor:
    """Compose signed text components with SLERP over positive and negative prompt vectors."""
    if not 0.0 <= float(negative_weight) <= 1.0:
        raise ValueError("negative_weight must be in [0, 1]")

    pieces = parse_query(query)
    has_pos = any(p.sign == "+" for p in pieces)
    has_neg = any(p.sign == "-" for p in pieces)
    vectors, weights = [], []

    for attr in pieces:
        if attr.sign == "+":
            vec = bank["positive"][attr.raw].to(device)
            weight = (1.0 - negative_weight) if has_neg else 1.0
        else:
            vec = bank["negative_basic"][attr.raw].to(device)
            weight = negative_weight if has_pos else 1.0
        vectors.append(normalize(vec))
        weights.append(float(weight))

    return spherical_weighted_average(vectors, weights)


def fuse_image_arithmetic(image_vec: torch.Tensor, text_vec: torch.Tensor, image_scale: float = 1.0) -> torch.Tensor:
    """Final arithmetic image fusion: normalize(image + text)."""
    return normalize(image_scale * image_vec + text_vec)


def fuse_image_slerp(image_vec: torch.Tensor, text_vec: torch.Tensor, image_alpha: float) -> torch.Tensor:
    """
    Final image/text SLERP.

    image_alpha=0.0 -> text direction; image_alpha=1.0 -> original image.
    """
    return slerp_vectors(text_vec, image_vec, alpha=image_alpha)


def fuse_text_image_slerp(text_features: torch.Tensor, reference_features: torch.Tensor, image_alpha: float) -> torch.Tensor:
    """Same image-heavy convention used by the neural prompt-search section."""
    return slerp_vectors(text_features, reference_features, alpha=image_alpha)


_TEXT_DIRECTION_CACHE: dict[tuple, torch.Tensor] = {}


def _text_cache_key(query: str, method: MethodSpec) -> tuple:
    return (query, method.text_mode, float(method.negative_weight))


def text_direction_for_method(query: str, method: MethodSpec, bank: dict) -> torch.Tensor:
    """Cache query-level text directions; full evaluation repeats each query many times."""
    key = _text_cache_key(query, method)
    if key not in _TEXT_DIRECTION_CACHE:
        if method.text_mode == "arithmetic":
            vec = text_direction_arithmetic(query, bank)
        elif method.text_mode == "slerp":
            vec = text_direction_slerp(query, bank, negative_weight=method.negative_weight)
        else:
            raise ValueError(f"Unknown text_mode: {method.text_mode}")
        _TEXT_DIRECTION_CACHE[key] = normalize(vec).detach()
    return _TEXT_DIRECTION_CACHE[key]


@torch.no_grad()
def make_modified_targets_batched(
    cases: list[RetrievalCase],
    method: MethodSpec,
    image_embeddings: ImageEmbeddings,
    bank: dict,
) -> torch.Tensor:
    """Build a [batch, dim] tensor of modified target vectors for one method."""
    if not cases:
        raise ValueError("No cases provided for batched target construction")
    if method.prediction_mode == "random":
        raise ValueError("Random baseline does not build modified target vectors")

    reference_ids = [case.reference_id for case in cases]
    image_vecs = image_embeddings.batch_get(reference_ids).to(device)
    text_vecs = torch.cat([
        text_direction_for_method(case.query, method, bank).reshape(1, -1)
        for case in cases
    ], dim=0).to(device)

    if method.fusion_mode == "add":
        return fuse_image_arithmetic(image_vecs, text_vecs, image_scale=method.image_scale)
    if method.fusion_mode == "slerp":
        return fuse_image_slerp(image_vecs, text_vecs, image_alpha=method.image_alpha)
    raise ValueError(f"Unknown fusion_mode: {method.fusion_mode}")


@torch.no_grad()
def predict_cases_batched(
    cases: list[RetrievalCase],
    method: MethodSpec,
    k: int,
    image_embeddings: ImageEmbeddings,
    bank: dict,
    batch_size: int = EVAL_BATCH_SIZE,
    exclude_reference: bool = EXCLUDE_REFERENCE,
    desc: Optional[str] = None,
    show_progress: bool = True,
) -> list[dict[int, float]]:
    """Predict top-k image IDs/scores for many cases using batched CUDA matmul."""
    if not cases:
        return []

    image_bank = image_embeddings.embeddings.to(device)
    image_ids = image_embeddings.ids
    predictions_by_case = []
    k_eff = min(int(k), image_bank.shape[0])

    starts = range(0, len(cases), batch_size)
    if show_progress:
        starts = tqdm(starts, desc=desc or method.name)

    for start in starts:
        batch_cases = cases[start:start + batch_size]

        if method.prediction_mode == "random":
            # Random scores are generated on the GPU so the baseline follows the same batched top-k path.
            scores = torch.rand((len(batch_cases), image_bank.shape[0]), device=device)
        else:
            modified_targets = make_modified_targets_batched(batch_cases, method, image_embeddings, bank)
            # One large [batch, dim] @ [dim, n_images] multiplication keeps CUDA busy.
            scores = modified_targets @ image_bank.T

        if exclude_reference:
            reference_ids = [case.reference_id for case in batch_cases]
            reference_rows = image_embeddings.rows_for_ids(reference_ids).to(scores.device)
            batch_rows = torch.arange(len(batch_cases), device=scores.device)
            scores[batch_rows, reference_rows] = -float("inf")

        values, rows = torch.topk(scores, k=k_eff, dim=1)
        values = values.detach().cpu()
        rows = rows.detach().cpu()

        for case_values, case_rows in zip(values, rows):
            predictions_by_case.append({
                image_ids[int(row)]: float(value)
                for row, value in zip(case_rows, case_values)
            })

    return predictions_by_case


@torch.no_grad()
def predict_case(
    case: RetrievalCase,
    method: MethodSpec,
    k: int,
    image_embeddings: ImageEmbeddings = image_embeddings,
    bank: dict = text_bank,
    exclude_reference: bool = EXCLUDE_REFERENCE,
) -> dict[int, float]:
    """Single-case wrapper around the same batched prediction path."""
    return predict_cases_batched(
        [case],
        method,
        k=k,
        image_embeddings=image_embeddings,
        bank=bank,
        batch_size=1,
        exclude_reference=exclude_reference,
        show_progress=False,
    )[0]


def evaluate_method_cases(
    method: MethodSpec,
    cases: list[RetrievalCase],
    ks: Sequence[int] = KS,
    image_embeddings: ImageEmbeddings = image_embeddings,
    bank: dict = text_bank,
    batch_size: int = EVAL_BATCH_SIZE,
    exclude_reference: bool = EXCLUDE_REFERENCE,
    show_progress: bool = True,
) -> pd.DataFrame:
    """Evaluate one method on every case and return one row per case."""
    predictions_by_case = predict_cases_batched(
        cases,
        method,
        k=max(ks),
        image_embeddings=image_embeddings,
        bank=bank,
        batch_size=batch_size,
        exclude_reference=exclude_reference,
        desc=method.name,
        show_progress=show_progress,
    )

    rows = []
    iterator = zip(cases, predictions_by_case)
    if show_progress:
        iterator = tqdm(iterator, total=len(cases), desc=f"Scoring {method.name}")

    for case, predictions in iterator:
        retrieved = list(predictions.keys())
        row = {
            **method_config_row(method),
            "query_id": case.query_id,
            "query": case.query,
            "reference_id": case.reference_id,
            "n_gt": len(case.ground_truth),
            "batch_size": batch_size,
            "exclude_reference": exclude_reference,
        }
        for k in ks:
            row.update(evaluate_retrieval(retrieved, case.ground_truth, k))
        rows.append(row)

    return pd.DataFrame(rows)


def evaluate_methods(
    methods: list[MethodSpec],
    cases: list[RetrievalCase],
    ks: Sequence[int] = KS,
    batch_size: int = EVAL_BATCH_SIZE,
    exclude_reference: bool = EXCLUDE_REFERENCE,
) -> pd.DataFrame:
    """Run several methods and concatenate their per-case result dataframes."""
    if not cases:
        raise ValueError("No evaluation cases provided")
    frames = []
    for method in methods:
        frames.append(evaluate_method_cases(
            method,
            cases,
            ks=ks,
            batch_size=batch_size,
            exclude_reference=exclude_reference,
        ))
    return pd.concat(frames, ignore_index=True)


def metric_columns(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if c.startswith(("Recall@", "Precision@"))]


def summarize_results_by_query(results_df: pd.DataFrame) -> pd.DataFrame:
    """Average metrics per query, preserving method columns when a section compares methods."""
    metrics = metric_columns(results_df)
    group_cols = ["method", "query_id", "query"] if "method" in results_df.columns else ["query_id", "query"]
    summary = results_df.groupby(group_cols, as_index=False).agg(
        n_cases=("reference_id", "count"),
        **{c: (c, "mean") for c in metrics},
    )
    return summary


def summarize_results_overall(results_df: pd.DataFrame) -> pd.DataFrame:
    """Optional compact method-level summary for ranking methods after the per-query table."""
    metrics = metric_columns(results_df)
    group_cols = ["method"] if "method" in results_df.columns else []
    if group_cols:
        return results_df.groupby(group_cols, as_index=False).agg(
            n_cases=("reference_id", "count"),
            **{c: (c, "mean") for c in metrics},
        )
    row = {"n_cases": len(results_df)}
    row.update({c: results_df[c].mean() for c in metrics})
    return pd.DataFrame([row])

In [11]:
# Full benchmark cases used by all quantitative sections.
# query_indices=None and max_references_per_query=None means the entire JSON benchmark.
FULL_CASES = make_cases(
    annotations,
    image_embeddings,
    query_indices=None,
    max_references_per_query=None,
)
print(f"Full evaluation cases: {len(FULL_CASES)}")

# A tiny case list is convenient for debugging individual methods without touching the full benchmark.
DEBUG_CASES = make_cases(
    annotations,
    image_embeddings,
    query_indices=[0] if annotations else None,
    max_references_per_query=3,
)
print(f"Debug cases: {len(DEBUG_CASES)}")
DEBUG_CASES[:2]

Full evaluation cases: 33052
Debug cases: 3


[RetrievalCase(query_id=0, query='+Smiling', reference_id=13, ground_truth=[325, 456, 579, 685, 763, 893, 981, 1363, 1646, 2142, 2747, 2812, 2979, 3318, 3325, 3536, 3637, 3932, 4361, 4452, 4549, 4569, 4678, 4780, 4831, 5112, 5208, 5448, 5533, 5926, 6127, 6153, 7595, 7798, 8151, 8342, 8440, 8592, 9009, 9452, 9506, 9612, 9685, 9881, 9930, 9968, 10367, 10719, 10915, 10968, 10973, 11813, 12361, 12673, 12695, 12788, 12934, 12986, 13060, 13141, 13731, 14215, 14323, 14848, 15058, 15422, 15953, 17205, 17384, 17638, 17748, 18096, 18202, 18221, 18290, 18446, 18503, 18799, 19046, 19179, 19800]),
 RetrievalCase(query_id=0, query='+Smiling', reference_id=14, ground_truth=[235, 1787, 2806, 16035, 19731])]

### Qualitative Plotting Helpers

In [12]:
celeba = None


def load_celeba_for_plots():
    """Load CelebA test images for qualitative plots, if available."""
    global celeba
    if celeba is not None:
        return celeba
    try:
        # download=False prevents torchvision from trying network access on the VM.
        celeba = CelebA(root=CELEBA_ROOT, split="test", download=False)
        print("CelebA test split loaded for plotting.")
    except Exception as exc:
        celeba = None
        print("CelebA images are not available; qualitative plots will show IDs only.")
        print("Reason:", exc)
    return celeba


def _show_image_id(ax, image_id: int, title: str):
    dataset = load_celeba_for_plots()
    if dataset is None:
        ax.text(0.5, 0.5, str(image_id), ha="center", va="center", fontsize=12)
        ax.set_title(title)
        ax.axis("off")
        return
    image, _ = dataset[int(image_id)]
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")


def plot_case_predictions(case: RetrievalCase, predictions: dict[int, float], k_to_plot: int = 5, title: str = "Predictions"):
    """Plot reference, a few ground-truth targets, and top predictions for one case."""
    n_gt = min(k_to_plot, len(case.ground_truth))
    n_pred = min(k_to_plot, len(predictions))
    n_cols = 1 + n_gt + n_pred
    fig, axes = plt.subplots(1, n_cols, figsize=(3.2 * n_cols, 3.6))
    if n_cols == 1:
        axes = [axes]

    _show_image_id(axes[0], case.reference_id, f"Reference {case.reference_id}\n{case.query}")
    offset = 1
    for ax, target_id in zip(axes[offset:offset + n_gt], case.ground_truth[:n_gt]):
        _show_image_id(ax, target_id, f"GT {target_id}")
    offset += n_gt
    for ax, (pred_id, score) in zip(axes[offset:offset + n_pred], list(predictions.items())[:n_pred]):
        _show_image_id(ax, pred_id, f"Pred {pred_id}\n{score:.3f}")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

---
## Mapping network
---

### Model

In [13]:
class IM2TEXT(nn.Module):
    """pic2word model (MLP network).

    Attributes:
        fc_out: output layer
        layers: layers of the network
    """
    def __init__(
            self,
            embed_dim=512,
            middle_dim=512,
            output_dim=512,
            n_layer=2,
            dropout=0.1
    ):
        """Initializes the model.

        Args:
            embed_dim: embedding dimension of images
            middle_dim: dimension of teh middle layer
            output_dim: dimension of the output layer
            n_layer: number of layers
            dropout: dropout rate
        """
        super().__init__()
        self.fc_out = nn.Linear(middle_dim, output_dim)
        layers = []
        dim = embed_dim
        for _ in range(n_layer):
            block = []
            block.append(nn.Linear(dim, middle_dim))
            block.append(nn.Dropout(dropout))
            block.append(nn.ReLU())
            dim = middle_dim
            layers.append(nn.Sequential(*block))
        self.layers = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor):
        """Forward method.

        Args:
            x: input tensor

        Returns:
            output tensor
        """
        for layer in self.layers:
            x = layer(x)
        return self.fc_out(x)

### Image and text features

In [29]:
def get_image_features(images):
    """Compute batch image features with the CLIP image encoder.
    
    Args:
        images: batch images (already processed by dataloader)
        
    Returns:
        batch image features
    """
    # encode the images directly with CLIP
    with torch.no_grad():
        images_z_output = model.get_image_features(pixel_values=images)

    images_z = images_z_output.pooler_output # the tensors

    return images_z

def get_normalized_image_features(images_features, device = "cuda"):
    """Return the image features normalized.

    Args:
        images_features: image features
        device: where to perform calculations

    Returns:
        batch image features normalized
    """
    images_z = images_features / images_features.norm(dim=-1, keepdim=True)

    return images_z

def encode_text_with_image(input_ids, img_tokens):
    """
    Injects an image token into CLIP text embeddings just before the End-of-Text
    (EOT) token assuming as text "a photo of".

    Args:
        model: a frozen Hugging Face CLIPModel
        input_ids: text token IDs, shape [batch_size, n_ctx] (typically n_ctx=77)
        img_tokens: image features to inject, shape [batch_size, hidden_size]

    Returns:
        Projected multimodal features, shape [batch_size, projection_dim]
    """
    batch_size = input_ids.size(0)

    # get the base token embeddings from CLIP
    with torch.no_grad():
        token_embeds = model.text_model.embeddings.token_embedding(input_ids)

    # find the EOT token index (usually the largest, the one at the end)
    eot_indices = input_ids.argmax(dim=-1)

    # take the join index from the first item in the batch
    splice_idx = eot_indices[0].item()

    # insert the image tokens into the embeddings
    img_tokens = img_tokens.view(batch_size, 1, -1)
    spliced_embeds = torch.cat([
        token_embeds[:, :splice_idx],
        img_tokens,
        token_embeds[:, splice_idx:-1]
    ], dim=1)

    # create a runtime patch for the embedding layer's forward method
    original_embeddings_forward = model.text_model.embeddings.forward

    def patched_embeddings_forward(
        input_ids=None,
        position_ids=None,
        inputs_embeds=None
    ):
        # we intercept the call, drop input_ids, and force feed our custom
        # token embeddings
        return original_embeddings_forward(
            input_ids=None,
            position_ids=position_ids,
            inputs_embeds=spliced_embeds
        )

    try:
        # apply the runtime patch
        model.text_model.embeddings.forward = patched_embeddings_forward

        # run high-level text model safely
        outputs = model.text_model(input_ids=input_ids)
        last_hidden_state = outputs.last_hidden_state
    finally:
        # restore original method to avoid breaking downstream text tracking
        model.text_model.embeddings.forward = original_embeddings_forward

    # extract features at the shifted EOS token position
    # (+1 due to the injection)
    new_eot_indices = eot_indices + 1
    batch_indices = torch.arange(batch_size, device=input_ids.device)
    eot_features = last_hidden_state[batch_indices, new_eot_indices]

    projected_features = model.text_projection(eot_features)

    return projected_features

def get_normalized_text_features(token_features, text_inputs, device = "cuda"):
    """Compute text features with the CLIP text encoder.

    Args:
        token_features: image features
        device: where to perform calculations

    Returns:
        text features
    """
    # extract input_ids
    input_ids = text_inputs["input_ids"].to(device)

    # repeat the tokenized row to match the batch size of token_features
    input_ids = input_ids.repeat(token_features.size(0), 1)  # Shape: [batch_size, 77]

    # features + pseudo language token
    text_features = encode_text_with_image(input_ids, token_features)

    # normalize the features
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    return text_features

def a_photo_of_tokens(device = "cuda"):
    """Returns text tokens of "a photo of" text.
    """
    text_inputs = processor(
        text="a photo of",
        padding="max_length",
        max_length=77,
        truncation=True,
        return_tensors="pt"
    )
    text_inputs = text_inputs.to(device)
    
    return text_inputs

### Data loaders

In [15]:
def get_celeba(root, batch_size, download=False):
    """Create celeba dataloaders.

    Args:
        root: path to the stored celeba data
        batch_size: how many samples to process in parallel. GPU-dependent
        download: whether to download the dataset or not

    Returns:
        training and validation data loaders
    """
    
    # replicate CLIP native image processing (specif of the model patch)
    transform = transforms.Compose([
        transforms.Resize((224, 224), interpolation=InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.48145466, 0.4578275, 0.40821073),
            std=(0.26862954, 0.26130258, 0.27577711)
        )
    ])

    training_data = CelebA(root=root, split="train", download=download, transform=transform)
    validation_data = CelebA(root=root, split="valid", download=download, transform=transform)

    print(f"# of training samples: {len(training_data)}")
    print(f"# of validation samples: {len(validation_data)}")

    train_loader = torch.utils.data.DataLoader(training_data, batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(validation_data, int(batch_size/2), shuffle=False, num_workers=4, pin_memory=True)

    return train_loader, val_loader

### Training

In [33]:
def get_optimizer(img2text, wd = 1e-4, lr = 1e-4):
    """Return the training (AdamW) optimizer of pic2word.

    Args:
        img2text: pic2word model
        wd: weight decay
        lr: learning rate

    Returns:
        optimizer
    """
    named_parameters = list(img2text.named_parameters())
    exclude = lambda n : "bn" in n or "ln" in n or "bias" in n or 'logit_scale' in n
    include = lambda n : not exclude(n)
    gain_or_bias_params = [p for n, p in named_parameters if exclude(n) and p.requires_grad]
    rest_params = [p for n, p in named_parameters if include(n) and p.requires_grad]

    optimizer = torch.optim.AdamW(
        [
            {"params": gain_or_bias_params, "weight_decay": 0.0},
            {"params": rest_params, "weight_decay": wd},
        ],
        lr=lr,
    )

    return optimizer

In [27]:
def get_loss_img2text(loss_img, loss_txt, images_features, token_features, text_inputs, device = "cuda"):
    """Get the training loss (contrastive loss) of pic2word

    Args:
        loss_img: loss function (cross entropy loss) for images
        loss_txt: loss function (cross entropy loss) for text
        images_features: image features
        token_features: text features of the embedded pseudo-language token
        device: where to perform calculations

    Returns:
        image features, text features and loss
    """
    image_features = get_normalized_image_features(images_features)
    text_features = get_normalized_text_features(token_features, text_inputs)

    logit_scale = model.logit_scale.exp()
    logit_scale = logit_scale.mean()

    ground_truth = torch.arange(len(image_features)).long()
    ground_truth = ground_truth.to(device)

    logits_per_image = logit_scale * image_features @ text_features.t()
    loss_img_val = loss_img(logits_per_image, ground_truth)
    logits_per_text = logit_scale * text_features @ image_features.t()
    loss_txt_val = loss_txt(logits_per_text, ground_truth)
    total_loss = (loss_img_val + loss_txt_val) / 2

    return image_features, text_features, total_loss

In [30]:
def training_step(
    img2text,
    training_data_loader,
    text_inputs,
    optimizer,
    loss_img,
    loss_txt,
    epoch,
    log_file,
    device="cuda",
    tb_writer=None
):
    """Training step of pic2word.

    Args:
        img2text: pic2word model
        training_data_loader: training data loader
        optimizer: training optimizer (AdamW)
        loss_img: loss function (cross entropy loss) for images
        loss_txt: loss function (cross entropy loss) for text
        epoch: current epoch
        device: where to perform calculations
        tb_writer: tensorboard writer
    """
    num_batches_per_epoch = len(training_data_loader)

    # set the network to training mode
    img2text.train()
    # iterate over the training set
    for i, (images, _) in enumerate(training_data_loader):
        # store the actual step of the training
        step = num_batches_per_epoch * epoch + i

        # reset gradients
        optimizer.zero_grad()

        # move input images to cuda
        images = images.to(device)
        # extract CLIP features
        clip_image_features = get_image_features(images)

        # forward pass: img2text processes the CLIP image features
        img_tokens = img2text(clip_image_features)

        # loss computation: pass raw images and img_tokens to the loss function
        image_features, text_features, loss = get_loss_img2text(
            loss_img,
            loss_txt,
            clip_image_features,
            img_tokens,
            text_inputs,
            device
        )

        # backward pass (compute the gradients)
        loss.backward()

        # parameters update
        optimizer.step()

        # every 64 batches print training evolution
        if (i%64) == 0:
            num_samples = i * len(images)
            samples_per_epoch = len(training_data_loader.dataset)
            percent_complete = 100.0 * i / num_batches_per_epoch
            # print status of training
            msg = (
                f"Train Epoch: {epoch} [{num_samples}/{samples_per_epoch} ({percent_complete:.0f}%)]\t"
                f"Loss: {loss.item():.6f}"
                f"\tLR: {optimizer.param_groups[0]['lr']:5f}\tlogit_scale {model.logit_scale.data:.3f}"
            )
            print(msg)
            # append status to the log file
            with open(log_file, "a") as f:
                f.write(msg + "\n")
                f.flush()

        # every 32 batches store loss updates (also scale and lr, that for now
        # are static) on the tensor board
        if (i%32) == 0:
            timestep = epoch * num_batches_per_epoch + i
            log_data = {
                "loss": loss.item(),
                "cossim": cosine_similarity_mean(text_features, image_features)
            }

            for name, val in log_data.items():
                name = "train/" + name
                if tb_writer is not None:
                    tb_writer.add_scalar(name, val, timestep)
                    

### Testing

In [19]:
def cosine_similarity_mean(text_pseudo_z, image_z):
    """Computes cosine similarity mean.

    Args:
        text_pseudo_z: text features modified with pseudo-language tokens
        image_z: normalized image features

    Returns:
        cosine similarity mean
    """
    return F.cosine_similarity(text_pseudo_z, image_z, dim=-1).mean().cpu()

In [20]:
def val_step(
    img2text,
    val_data_loader,
    text_inputs,
    loss_img,
    loss_txt,
    epoch,
    log_file,
    tb_writer=None,
    device="cuda",
):
    num_batches_per_epoch = len(val_data_loader)

    # set the network to validation/testing mode
    img2text.eval()

    # iterate over the training set
    with torch.no_grad():
      for i, (images, _) in enumerate(val_data_loader):
          # store the actual step of the training
          step = num_batches_per_epoch * epoch + i
          # move input images to cuda
          images = images.to(device)
          # extract CLIP features from the raw images
          # extract CLIP features (images are already preprocessed and on device)
          clip_image_features = get_image_features(images)
          # obtain pseudo-language tokens
          img_tokens = img2text(clip_image_features)
          # loss computation: pass raw images and img_tokens to the loss function
          image_features, text_features, loss = get_loss_img2text(
              loss_img,
              loss_txt,
              clip_image_features,
              img_tokens,
              text_inputs,
              device
          )

          # every 8 batches print validation evolution
          if (i%8) == 0:
              num_samples = i * len(images)
              samples_per_epoch = len(val_data_loader.dataset)
              percent_complete = 100.0 * i / num_batches_per_epoch
              # print status of training
              msg = (
                  f"Validation Epoch: {epoch} [{num_samples}/{samples_per_epoch} ({percent_complete:.0f}%)]\t"
              )
              print(msg)
              # append status to the log file
              with open(log_file, "a") as f:
                  f.write(msg + "\n")
                  f.flush()
                  
              timestep = epoch * num_batches_per_epoch + i
              log_data = {
                  "loss": loss.item(),
                  "cossim": cosine_similarity_mean(text_features, image_features)
              }

              for name, val in log_data.items():
                  name = "val/" + name
                  if tb_writer is not None:
                      tb_writer.add_scalar(name, val, timestep)

### Main 

In [34]:
def main(
    exp_name="exp",
    root=Path("/content/datasets"),
    batch_size=256,     # how many samples to process in parallel. GPU-dependent
    device="cuda",    # Where to perform calculations
    wd = 1e-4,
    lr = 1e-4,
    epochs=10          # How many times to iterate over the entire train set
):
    # create a logger for the experiment
    writer = SummaryWriter(log_dir=f"runs/{exp_name}")
    log_file = f"runs/{exp_name}.log"

    # get dataloaders
    train_loader, val_loader = get_celeba(root, batch_size)
    text_inputs = a_photo_of_tokens()

    # instantiate the network and move it to the chosen device (GPU)
    img2text = IM2TEXT().to(device)

    # "print" the network to view all the modules, so its architecture
    print(img2text)

    # instantiate the optimizer
    optimizer = get_optimizer(img2text, wd, lr)

    # create loss functions and move them to the GPU
    loss_img = nn.CrossEntropyLoss().to(device)
    loss_txt = nn.CrossEntropyLoss().to(device)

    print("Training:")
    # for each epoch, train the network and then compute evaluation results
    for e in range(epochs):
        training_step(
            img2text,
            train_loader,
            text_inputs,
            optimizer,
            loss_img,
            loss_txt,
            e,
            log_file,
            device="cuda",
            tb_writer=writer
        )
        val_step(img2text, val_loader, text_inputs, loss_img, loss_txt, e, log_file, writer)

    # closes the logger
    writer.close()
    # store the trained model
    torch.save(img2text.state_dict(), f"runs/model_{exp_name}.pth")

    return img2text

In [ ]:
# net = main()
net = main(exp_name="exp", root=Path("/home/disi/deep_learning/DeepLearning2026/data/celeba"))

# of training samples: 162770
# of validation samples: 19867
IM2TEXT(
  (fc_out): Linear(in_features=512, out_features=512, bias=True)
  (layers): Sequential(
    (0): Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): Dropout(p=0.1, inplace=False)
      (2): ReLU()
    )
    (1): Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): Dropout(p=0.1, inplace=False)
      (2): ReLU()
    )
  )
)
Training:
Train Epoch: 0 [0/162770 (0%)]	Loss: 6.295677	LR: 0.000100	logit_scale 4.605
Train Epoch: 0 [16384/162770 (10%)]	Loss: 0.898748	LR: 0.000100	logit_scale 4.605


### Results

---
## Baselines
---

We start with the original signed arithmetic combination of image and text attributes, plus a random retrieval baseline that samples from the image bank.

In [13]:
BASELINE_METHODS = [
    MethodSpec(
        name="00 Random baseline",
        prediction_mode="random",
    ),
    MethodSpec(
        name="01 Arithmetic text + image",
        prediction_mode="composition",
        text_mode="arithmetic",
        fusion_mode="add",
    ),
]

baseline_registry_df = pd.DataFrame([method_config_row(method) for method in BASELINE_METHODS])
baseline_registry_df

,method,prediction_mode,text_mode,fusion_mode,negative_weight,image_alpha,image_scale
0,00 Random baseline,random,NaN,NaN,None,None,NaN
1,01 Arithmetic text + image,composition,arithmetic,add,None,None,1.0


In [14]:
baseline_results_df = evaluate_methods(
    BASELINE_METHODS,
    FULL_CASES,
    ks=KS,
    batch_size=EVAL_BATCH_SIZE,
    exclude_reference=EXCLUDE_REFERENCE,
)

baseline_query_table = summarize_results_by_query(baseline_results_df)
baseline_overall_table = summarize_results_overall(baseline_results_df)
baseline_query_table

00 Random baseline:   0%|          | 0/65 [00:00<?, ?it/s]

Scoring 00 Random baseline:   0%|          | 0/33052 [00:00<?, ?it/s]

01 Arithmetic text + image:   0%|          | 0/65 [00:00<?, ?it/s]

Scoring 01 Arithmetic text + image:   0%|          | 0/33052 [00:00<?, ?it/s]

,method,query_id,query,n_cases,Recall@1,Precision@1,Recall@5,Precision@5,Recall@10,Precision@10
0,00 Random baseline,0,+Smiling,4786,0.001254,0.001254,0.007940,0.001630,0.012954,0.001337
1,00 Random baseline,1,+Eyeglasses,2196,0.001366,0.001366,0.005009,0.001002,0.009107,0.000911
2,00 Random baseline,2,-Heavy_Makeup,4087,0.001223,0.001223,0.004160,0.000832,0.008808,0.000881
3,00 Random baseline,3,+Male,1595,0.002508,0.002508,0.014420,0.003009,0.028840,0.003009
4,00 Random baseline,4,-Young,5355,0.001307,0.001307,0.005789,0.001195,0.011578,0.001214
5,00 Random baseline,5,+Blond_Hair,5469,0.001646,0.001646,0.008045,0.001609,0.014262,0.001463
6,00 Random baseline,6,+Mustache,301,0.000000,0.000000,0.000000,0.000000,0.003322,0.000332
7,00 Random baseline,7,-Young,5355,0.001307,0.001307,0.005602,0.001158,0.012138,0.001232
8,00 Random baseline,8,"+Eyeglasses, +Smiling",612,0.001634,0.001634,0.004902,0.000980,0.006536,0.000654
9,00 Random baseline,9,"+Black_Hair, -Wavy_Hair",2572,0.002333,0.002333,0.006221,0.001244,0.013997,0.001439


In [15]:
# # Qualitative baseline example. Uncomment only when you want images/IDs in the notebook output.
# baseline_case = FULL_CASES[0]
# for method in BASELINE_METHODS:
#     baseline_predictions = predict_case(baseline_case, method, k=5)
#     plot_case_predictions(
#         baseline_case,
#         baseline_predictions,
#         k_to_plot=5,
#         title=method.name,
#     )

---
## Simple SLERP Image Fusion
---

Here the signed text direction remains the arithmetic baseline, but the final fusion between the text direction and the reference image embedding is spherical instead of additive.

In [16]:
SIMPLE_SLERP_METHODS = [
    MethodSpec(
        name="03 SLERP image fusion, image-heavy",
        prediction_mode="composition",
        text_mode="arithmetic",
        fusion_mode="slerp",
        image_alpha=0.8,
    ),
]

simple_slerp_registry_df = pd.DataFrame([method_config_row(method) for method in SIMPLE_SLERP_METHODS])
simple_slerp_registry_df

,method,prediction_mode,text_mode,fusion_mode,negative_weight,image_alpha,image_scale
0,"03 SLERP image fusion, image-heavy",composition,arithmetic,slerp,None,0.8,None


In [17]:
simple_slerp_results_df = evaluate_methods(
    SIMPLE_SLERP_METHODS,
    FULL_CASES,
    ks=KS,
    batch_size=EVAL_BATCH_SIZE,
    exclude_reference=EXCLUDE_REFERENCE,
)

simple_slerp_query_table = summarize_results_by_query(simple_slerp_results_df)
simple_slerp_overall_table = summarize_results_overall(simple_slerp_results_df)
simple_slerp_query_table

03 SLERP image fusion, image-heavy:   0%|          | 0/65 [00:00<?, ?it/s]

Scoring 03 SLERP image fusion, image-heavy:   0%|          | 0/33052 [00:00<?, ?it/s]

,method,query_id,query,n_cases,Recall@1,Precision@1,Recall@5,Precision@5,Recall@10,Precision@10
0,"03 SLERP image fusion, image-heavy",0,+Smiling,4786,0.045550,0.045550,0.153991,0.039532,0.224613,0.034204
1,"03 SLERP image fusion, image-heavy",1,+Eyeglasses,2196,0.003643,0.003643,0.028233,0.006648,0.056011,0.007514
2,"03 SLERP image fusion, image-heavy",2,-Heavy_Makeup,4087,0.027404,0.027404,0.088084,0.020064,0.135307,0.017152
3,"03 SLERP image fusion, image-heavy",3,+Male,1595,0.005016,0.005016,0.011912,0.003386,0.021944,0.003386
4,"03 SLERP image fusion, image-heavy",4,-Young,5355,0.009897,0.009897,0.030065,0.007246,0.045938,0.006256
5,"03 SLERP image fusion, image-heavy",5,+Blond_Hair,5469,0.019931,0.019931,0.058512,0.015908,0.083196,0.013513
6,"03 SLERP image fusion, image-heavy",6,+Mustache,301,0.073090,0.073090,0.152824,0.041860,0.202658,0.031229
7,"03 SLERP image fusion, image-heavy",7,-Young,5355,0.009897,0.009897,0.030065,0.007246,0.045938,0.006256
8,"03 SLERP image fusion, image-heavy",8,"+Eyeglasses, +Smiling",612,0.001634,0.001634,0.016340,0.003268,0.026144,0.003105
9,"03 SLERP image fusion, image-heavy",9,"+Black_Hair, -Wavy_Hair",2572,0.011275,0.011275,0.048989,0.011198,0.078538,0.010692


In [18]:
# # Qualitative simple-SLERP example. Uncomment only when you want images/IDs in the notebook output.
# simple_slerp_case = FULL_CASES[0]
# simple_slerp_predictions = predict_case(simple_slerp_case, SIMPLE_SLERP_METHODS[0], k=5)
# plot_case_predictions(
#     simple_slerp_case,
#     simple_slerp_predictions,
#     k_to_plot=5,
#     title=SIMPLE_SLERP_METHODS[0].name,
# )

### SLERP variants

SLERP's promising results prompted us to explore other ways in which this composition could be exploited. In particular, we wanted to answer the following questions:

- *could SLERP beat the baseline as the image fusion mechanism? What about combining attributes?*
- *should the image embedding be given more or less weight in the fusion mechanism?*
- *with a SLERP fusion mechanism, should we give more or less weight to negative prompts?*

We explored these questions by creating a series of test cases that test every combination of these hypotheses. In particular we tested:
- **text composition**: signed arithmetic vs SLERP

- **image fusion mechanism**: arithmetic add vs SLERP

When testing the SLERP textual composition method, we tested a balaced approach, as well as giving more weight to negative attributes, and more weight to the positive attributes. Negative attributes are harder to treat in these contexts, thus this test case.

When testing the SLERP image fusion, we considered a balanced approach, as well as beingier on the image embedding and being heavier on the text embedding. The intuition here is that a heavier $\alpha$ on the image vector results in a final embedding closer to the image, which should return a better results since image vectors lie close to each other in the CLIP embedding space.

The final table encopassing all these test cases is the following:

| # | Method | Text composition | Image fusion | Main knobs |
|---:|---|---|---|---|
| 01 | Arithmetic text + image | Signed arithmetic: `+positive -negative` | Arithmetic add: `normalize(image + text)` | Baseline behavior |
| 02 | SLERP image fusion, balanced | Signed arithmetic | SLERP: `slerp(text, image)` | `image_alpha=0.5` |
| 03 | SLERP image fusion, image-heavy | Signed arithmetic | SLERP: `slerp(text, image)` | `image_alpha=0.8` |
| 04 | SLERP image fusion, text-heavy | Signed arithmetic | SLERP: `slerp(text, image)` | `image_alpha=0.2` |
| 05 | Text SLERP + image | SLERP over signed text prompts | Arithmetic add | `negative_weight=0.5` |
| 06 | Negative-heavy text SLERP + image | SLERP over signed text prompts | Arithmetic add | `negative_weight=0.8` |
| 07 | Text-light text SLERP + image | SLERP over signed text prompts | Arithmetic add | `negative_weight=0.2` |
| 08 | Text SLERP + SLERP image fusion, balanced/balanced | SLERP over signed text prompts | SLERP: `slerp(text, image)` | `negative_weight=0.5`, `image_alpha=0.5` |
| 09 | Text SLERP + SLERP image fusion, balanced/image-heavy | SLERP over signed text prompts | SLERP: `slerp(text, image)` | `negative_weight=0.5`, `image_alpha=0.8` |
| 10 | Text SLERP + SLERP image fusion, balanced/text-heavy | SLERP over signed text prompts | SLERP: `slerp(text, image)` | `negative_weight=0.5`, `image_alpha=0.2` |
| 11 | Text SLERP + SLERP image fusion, negative-heavy/balanced | SLERP over signed text prompts | SLERP: `slerp(text, image)` | `negative_weight=0.8`, `image_alpha=0.5` |
| 12 | Text SLERP + SLERP image fusion, negative-heavy/image-heavy | SLERP over signed text prompts | SLERP: `slerp(text, image)` | `negative_weight=0.8`, `image_alpha=0.8` |
| 13 | Text SLERP + SLERP image fusion, negative-heavy/text-heavy | SLERP over signed text prompts | SLERP: `slerp(text, image)` | `negative_weight=0.8`, `image_alpha=0.2` |
| 14 | Text SLERP + SLERP image fusion, text-light/balanced | SLERP over signed text prompts | SLERP: `slerp(text, image)` | `negative_weight=0.2`, `image_alpha=0.5` |
| 15 | Text SLERP + SLERP image fusion, text-light/image-heavy | SLERP over signed text prompts | SLERP: `slerp(text, image)` | `negative_weight=0.2`, `image_alpha=0.8` |
| 16 | Text SLERP + SLERP image fusion, text-light/text-heavy | SLERP over signed text prompts | SLERP: `slerp(text, image)` | `negative_weight=0.2`, `image_alpha=0.2` |


In [19]:
MULTIPLE_SLERP_METHODS = [
    # Signed-arithmetic baseline and image-fusion ablations.
    MethodSpec(
        name="01 Arithmetic text + image",
        text_mode="arithmetic",
        fusion_mode="add",
    ),
    MethodSpec(
        name="02 SLERP image fusion, balanced",
        text_mode="arithmetic",
        fusion_mode="slerp",
        image_alpha=0.5,
    ),
    MethodSpec(
        name="03 SLERP image fusion, image-heavy",
        text_mode="arithmetic",
        fusion_mode="slerp",
        image_alpha=0.8,
    ),
    MethodSpec(
        name="04 SLERP image fusion, text-heavy",
        text_mode="arithmetic",
        fusion_mode="slerp",
        image_alpha=0.2,
    ),

    # Text-SLERP with arithmetic image fusion.
    MethodSpec(
        name="05 Text SLERP + image",
        text_mode="slerp",
        fusion_mode="add",
        negative_weight=0.5,
    ),
    MethodSpec(
        name="06 Negative-heavy text SLERP + image",
        text_mode="slerp",
        fusion_mode="add",
        negative_weight=0.8,
    ),
    MethodSpec(
        name="07 Text-light text SLERP + image",
        text_mode="slerp",
        fusion_mode="add",
        negative_weight=0.2,
    ),

    # Text-SLERP weights crossed with image/text SLERP alpha.
    MethodSpec(
        name="08 Text SLERP + SLERP image fusion, balanced/balanced",
        text_mode="slerp",
        fusion_mode="slerp",
        negative_weight=0.5,
        image_alpha=0.5,
    ),
    MethodSpec(
        name="09 Text SLERP + SLERP image fusion, balanced/image-heavy",
        text_mode="slerp",
        fusion_mode="slerp",
        negative_weight=0.5,
        image_alpha=0.8,
    ),
    MethodSpec(
        name="10 Text SLERP + SLERP image fusion, balanced/text-heavy",
        text_mode="slerp",
        fusion_mode="slerp",
        negative_weight=0.5,
        image_alpha=0.2,
    ),
    MethodSpec(
        name="11 Text SLERP + SLERP image fusion, negative-heavy/balanced",
        text_mode="slerp",
        fusion_mode="slerp",
        negative_weight=0.8,
        image_alpha=0.5,
    ),
    MethodSpec(
        name="12 Text SLERP + SLERP image fusion, negative-heavy/image-heavy",
        text_mode="slerp",
        fusion_mode="slerp",
        negative_weight=0.8,
        image_alpha=0.8,
    ),
    MethodSpec(
        name="13 Text SLERP + SLERP image fusion, negative-heavy/text-heavy",
        text_mode="slerp",
        fusion_mode="slerp",
        negative_weight=0.8,
        image_alpha=0.2,
    ),
    MethodSpec(
        name="14 Text SLERP + SLERP image fusion, text-light/balanced",
        text_mode="slerp",
        fusion_mode="slerp",
        negative_weight=0.2,
        image_alpha=0.5,
    ),
    MethodSpec(
        name="15 Text SLERP + SLERP image fusion, text-light/image-heavy",
        text_mode="slerp",
        fusion_mode="slerp",
        negative_weight=0.2,
        image_alpha=0.8,
    ),
    MethodSpec(
        name="16 Text SLERP + SLERP image fusion, text-light/text-heavy",
        text_mode="slerp",
        fusion_mode="slerp",
        negative_weight=0.2,
        image_alpha=0.2,
    ),
]

multiple_slerp_registry_df = pd.DataFrame([method_config_row(method) for method in MULTIPLE_SLERP_METHODS])
multiple_slerp_registry_df

,method,prediction_mode,text_mode,fusion_mode,negative_weight,image_alpha,image_scale
0,01 Arithmetic text + image,composition,arithmetic,add,NaN,NaN,1.0
1,"02 SLERP image fusion, balanced",composition,arithmetic,slerp,NaN,0.5,NaN
2,"03 SLERP image fusion, image-heavy",composition,arithmetic,slerp,NaN,0.8,NaN
3,"04 SLERP image fusion, text-heavy",composition,arithmetic,slerp,NaN,0.2,NaN
4,05 Text SLERP + image,composition,slerp,add,0.5,NaN,1.0
5,06 Negative-heavy text SLERP + image,composition,slerp,add,0.8,NaN,1.0
6,07 Text-light text SLERP + image,composition,slerp,add,0.2,NaN,1.0
7,"08 Text SLERP + SLERP image fusion, balanced/b...",composition,slerp,slerp,0.5,0.5,NaN
8,"09 Text SLERP + SLERP image fusion, balanced/i...",composition,slerp,slerp,0.5,0.8,NaN
9,"10 Text SLERP + SLERP image fusion, balanced/t...",composition,slerp,slerp,0.5,0.2,NaN


In [20]:
multiple_slerp_results_df = evaluate_methods(
    MULTIPLE_SLERP_METHODS,
    FULL_CASES,
    ks=KS,
    batch_size=EVAL_BATCH_SIZE,
    exclude_reference=EXCLUDE_REFERENCE,
)

multiple_slerp_query_table = summarize_results_by_query(multiple_slerp_results_df)
multiple_slerp_overall_table = summarize_results_overall(multiple_slerp_results_df)
multiple_slerp_query_table

01 Arithmetic text + image:   0%|          | 0/65 [00:00<?, ?it/s]

Scoring 01 Arithmetic text + image:   0%|          | 0/33052 [00:00<?, ?it/s]

02 SLERP image fusion, balanced:   0%|          | 0/65 [00:00<?, ?it/s]

Scoring 02 SLERP image fusion, balanced:   0%|          | 0/33052 [00:00<?, ?it/s]

03 SLERP image fusion, image-heavy:   0%|          | 0/65 [00:00<?, ?it/s]

...

15 Text SLERP + SLERP image fusion, text-light/image-heavy:   0%|          | 0/65 [00:00<?, ?it/s]

Scoring 15 Text SLERP + SLERP image fusion, text-light/image-heavy:   0%|          | 0/33052 [00:00<?, ?it/s]

16 Text SLERP + SLERP image fusion, text-light/text-heavy:   0%|          | 0/65 [00:00<?, ?it/s]

Scoring 16 Text SLERP + SLERP image fusion, text-light/text-heavy:   0%|          | 0/33052 [00:00<?, ?it/s]

,method,query_id,query,n_cases,Recall@1,Precision@1,Recall@5,Precision@5,Recall@10,Precision@10
0,01 Arithmetic text + image,0,+Smiling,4786,0.045341,0.045341,0.146678,0.037693,0.221270,0.033264
1,01 Arithmetic text + image,1,+Eyeglasses,2196,0.003643,0.003643,0.023224,0.005556,0.042805,0.005328
2,01 Arithmetic text + image,2,-Heavy_Makeup,4087,0.023978,0.023978,0.077563,0.017372,0.120382,0.014681
3,01 Arithmetic text + image,3,+Male,1595,0.005643,0.005643,0.017555,0.004138,0.026959,0.004013
4,01 Arithmetic text + image,4,-Young,5355,0.009897,0.009897,0.032493,0.007918,0.050794,0.006947
...,...,...,...,...,...,...,...,...,...,...
219,"16 Text SLERP + SLERP image fusion, text-light...",9,"+Black_Hair, -Wavy_Hair",2572,0.005443,0.005443,0.022162,0.005132,0.035770,0.004277
220,"16 Text SLERP + SLERP image fusion, text-light...",10,"-Male, -Mustache",27,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
221,"16 Text SLERP + SLERP image fusion, text-light...",11,"+Chubby, -Young",584,0.001712,0.001712,0.003425,0.000685,0.006849,0.000685
222,"16 Text SLERP + SLERP image fusion, text-light...",12,"-Smiling, +Eyeglasses, +Wearing_Hat",79,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [21]:
# # Qualitative multiple-SLERP example. Uncomment only when you want images/IDs in the notebook output.
# multiple_slerp_case = FULL_CASES[0]
# for method in MULTIPLE_SLERP_METHODS:
#     multiple_slerp_predictions = predict_case(multiple_slerp_case, method, k=5)
#     plot_case_predictions(
#         multiple_slerp_case,
#         multiple_slerp_predictions,
#         k_to_plot=5,
#         title=method.name,
#     )

---
## Selected Double-SLERP Backbone
---

From the grid above, the double-SLERP, image-heavy setting is the most useful backbone for the learned stage: text composition stays on the CLIP hypersphere, while the final target remains close to the reference image neighborhood. We therefore fix the geometry to balanced text SLERP with image-heavy fusion, `negative_weight=0.5` and `image_alpha=0.8`, and let the neural prompt-search stage improve the learned prompt/adaptor structure on top of that fixed choice.

In [22]:
SELECTED_SLERP_METHOD = MethodSpec(
    name="09 Text SLERP + SLERP image fusion, balanced/image-heavy",
    text_mode="slerp",
    fusion_mode="slerp",
    negative_weight=0.5,
    image_alpha=0.8,
)

selected_slerp_results_df = evaluate_methods(
    [SELECTED_SLERP_METHOD],
    FULL_CASES,
    ks=KS,
    batch_size=EVAL_BATCH_SIZE,
    exclude_reference=EXCLUDE_REFERENCE,
)

selected_slerp_query_table = summarize_results_by_query(selected_slerp_results_df)
selected_slerp_overall_table = summarize_results_overall(selected_slerp_results_df)
selected_slerp_query_table

09 Text SLERP + SLERP image fusion, balanced/image-heavy:   0%|          | 0/65 [00:00<?, ?it/s]

Scoring 09 Text SLERP + SLERP image fusion, balanced/image-heavy:   0%|          | 0/33052 [00:00<?, ?it/s]

,method,query_id,query,n_cases,Recall@1,Precision@1,Recall@5,Precision@5,Recall@10,Precision@10
0,"09 Text SLERP + SLERP image fusion, balanced/i...",0,+Smiling,4786,0.045550,0.045550,0.154827,0.040159,0.227748,0.034664
1,"09 Text SLERP + SLERP image fusion, balanced/i...",1,+Eyeglasses,2196,0.003643,0.003643,0.028689,0.006740,0.057377,0.007741
2,"09 Text SLERP + SLERP image fusion, balanced/i...",2,-Heavy_Makeup,4087,0.026181,0.026181,0.091754,0.021091,0.140935,0.017837
3,"09 Text SLERP + SLERP image fusion, balanced/i...",3,+Male,1595,0.005016,0.005016,0.012539,0.003511,0.020063,0.003072
4,"09 Text SLERP + SLERP image fusion, balanced/i...",4,-Young,5355,0.009150,0.009150,0.025584,0.006162,0.044444,0.005976
5,"09 Text SLERP + SLERP image fusion, balanced/i...",5,+Blond_Hair,5469,0.019931,0.019931,0.059426,0.015944,0.083745,0.013586
6,"09 Text SLERP + SLERP image fusion, balanced/i...",6,+Mustache,301,0.073090,0.073090,0.152824,0.041860,0.202658,0.030897
7,"09 Text SLERP + SLERP image fusion, balanced/i...",7,-Young,5355,0.009150,0.009150,0.025584,0.006162,0.044444,0.005976
8,"09 Text SLERP + SLERP image fusion, balanced/i...",8,"+Eyeglasses, +Smiling",612,0.001634,0.001634,0.016340,0.003268,0.027778,0.003268
9,"09 Text SLERP + SLERP image fusion, balanced/i...",9,"+Black_Hair, -Wavy_Hair",2572,0.013219,0.013219,0.048600,0.011353,0.083593,0.011198


In [23]:
# # Qualitative selected-backbone example. Uncomment only when you want images/IDs in the notebook output.
# selected_slerp_case = FULL_CASES[0]
# selected_slerp_predictions = predict_case(selected_slerp_case, SELECTED_SLERP_METHOD, k=5)
# plot_case_predictions(
#     selected_slerp_case,
#     selected_slerp_predictions,
#     k_to_plot=5,
#     title=SELECTED_SLERP_METHOD.name,
# )

---
## Neural Prompt Search Finetuning
---

This section brings in the updated SLERP finetuning idea: a CoOp-like signed prompt learner plus small searchable residual modules around the reference, text, and fused vectors. The SLERP geometry is fixed to the selected image-heavy double-SLERP backbone above; the search focuses on context length and lightweight adapters/low-rank residuals.

For an honest finetuning protocol, the reference cases are split by query into train, validation, and held-out test subsets. The supernet is trained on the train split, candidate architecture/knobs are selected on validation, and the reported neural prompt-search table uses only the held-out test split.

In [24]:
def expand_attention_mask(attention_mask, dtype):
    if attention_mask is None:
        return None
    expanded_mask = attention_mask[:, None, None, :].to(dtype=dtype)
    return (1.0 - expanded_mask) * torch.finfo(dtype).min


def causal_attention_mask(batch_size, sequence_length, dtype, device):
    mask = torch.full((batch_size, sequence_length, sequence_length), torch.finfo(dtype).min, dtype=dtype, device=device)
    mask = torch.triu(mask, diagonal=1)
    return mask[:, None, :, :]


def clip_text_features_from_embeddings(input_embeddings, input_ids, attention_mask=None):
    """Run frozen CLIP text transformer from custom input embeddings."""
    text_model = model.text_model
    text_device = next(text_model.parameters()).device
    text_dtype = next(text_model.parameters()).dtype

    input_embeddings = input_embeddings.to(device=text_device, dtype=text_dtype)
    input_ids = input_ids.to(text_device)
    attention_mask = None if attention_mask is None else attention_mask.to(text_device)

    batch_size, sequence_length, _ = input_embeddings.shape
    position_ids = torch.arange(sequence_length, device=text_device).unsqueeze(0)
    hidden_states = input_embeddings + text_model.embeddings.position_embedding(position_ids)

    encoder_kwargs = dict(
        inputs_embeds=hidden_states,
        attention_mask=expand_attention_mask(attention_mask, text_dtype),
        return_dict=True,
    )
    try:
        encoder_outputs = text_model.encoder(
            **encoder_kwargs,
            causal_attention_mask=causal_attention_mask(batch_size, sequence_length, text_dtype, text_device),
        )
    except TypeError:
        # Some transformers versions pass the causal mask internally.
        encoder_outputs = text_model.encoder(**encoder_kwargs)

    last_hidden_state = text_model.final_layer_norm(encoder_outputs.last_hidden_state)
    eos_positions = input_ids.argmax(dim=-1)
    pooled = last_hidden_state[torch.arange(batch_size, device=text_device), eos_positions]
    text_features = model.text_projection(pooled)
    return normalize(text_features.float())


class SignedCoOpPromptLearner(nn.Module):
    """Learns separate continuous prompt tokens for positive and negative attribute prompts."""

    def __init__(self, attribute_names: list[str], max_ctx: int = 12, ctx_init: str = "a photo of a"):
        super().__init__()
        self.attribute_names = list(attribute_names)
        self.attr_to_idx = {raw.lower(): i for i, raw in enumerate(self.attribute_names)}
        self.max_ctx = int(max_ctx)
        self.hidden_size = TEXT_WIDTH

        self.positive_ctx = nn.Parameter(self._init_context(len(attribute_names), max_ctx, ctx_init))
        self.negative_ctx = nn.Parameter(self._init_context(len(attribute_names), max_ctx, ctx_init))

        self._register_prompt_buffers("positive", [positive_suffix(raw) for raw in attribute_names])
        self._register_prompt_buffers("negative", [negative_suffix(raw) for raw in attribute_names])

    def _init_context(self, n_attr: int, max_ctx: int, ctx_init: str) -> torch.Tensor:
        dtype = next(model.text_model.parameters()).dtype
        if ctx_init:
            tokenized = tokenizer(ctx_init, return_tensors="pt", padding=False, truncation=True).to(device)
            with torch.no_grad():
                init = model.text_model.embeddings.token_embedding(tokenized["input_ids"])[0]
            valid = int(tokenized.get("attention_mask", torch.ones_like(tokenized["input_ids"]))[0].sum().item())
            init = init[:valid][1:-1]
        else:
            init = torch.empty(0, self.hidden_size, device=device, dtype=dtype)
        if init.shape[0] >= max_ctx:
            base = init[:max_ctx]
        else:
            pad = torch.empty(max_ctx - init.shape[0], self.hidden_size, device=device, dtype=dtype)
            nn.init.normal_(pad, std=0.02)
            base = torch.cat([init, pad], dim=0)
        return base.unsqueeze(0).repeat(n_attr, 1, 1).clone().detach().float()

    def _register_prompt_buffers(self, name: str, suffix_texts: list[str]) -> None:
        placeholder = " ".join(["X"] * self.max_ctx)
        prompt_texts = [f"{placeholder} {suffix}" for suffix in suffix_texts]
        tokenized = tokenizer(prompt_texts, padding="max_length", max_length=MAX_TEXT_LEN, truncation=True, return_tensors="pt")
        input_ids = tokenized["input_ids"].to(device)
        attention_mask = tokenized["attention_mask"].to(device)
        with torch.no_grad():
            token_embeddings = model.text_model.embeddings.token_embedding(input_ids)

        self.register_buffer(f"{name}_prefix_embeddings", token_embeddings[:, :1, :].float())
        self.register_buffer(f"{name}_suffix_embeddings", token_embeddings[:, 1 + self.max_ctx :, :].float())
        self.register_buffer(f"{name}_prefix_ids", input_ids[:, :1])
        self.register_buffer(f"{name}_suffix_ids", input_ids[:, 1 + self.max_ctx :])
        self.register_buffer(f"{name}_suffix_attention", attention_mask[:, 1 + self.max_ctx :])

    def _features_for_sign(self, sign_name: str, active_ctx: int) -> torch.Tensor:
        active_ctx = int(active_ctx)
        if not 1 <= active_ctx <= self.max_ctx:
            raise ValueError(f"active_ctx must be in [1, {self.max_ctx}], got {active_ctx}")
        ctx = self.positive_ctx if sign_name == "positive" else self.negative_ctx
        prefix_emb = getattr(self, f"{sign_name}_prefix_embeddings")
        suffix_emb = getattr(self, f"{sign_name}_suffix_embeddings")
        prefix_ids = getattr(self, f"{sign_name}_prefix_ids")
        suffix_ids = getattr(self, f"{sign_name}_suffix_ids")
        suffix_attention = getattr(self, f"{sign_name}_suffix_attention")

        batch = len(self.attribute_names)
        active_ctx_emb = ctx[:, :active_ctx, :]
        prompt_embeddings = torch.cat([prefix_emb, active_ctx_emb, suffix_emb], dim=1)
        ctx_ids = torch.zeros(batch, active_ctx, dtype=prefix_ids.dtype, device=prefix_ids.device)
        input_ids = torch.cat([prefix_ids, ctx_ids, suffix_ids], dim=1)
        prefix_attention = torch.ones(batch, 1, dtype=suffix_attention.dtype, device=suffix_attention.device)
        ctx_attention = torch.ones(batch, active_ctx, dtype=suffix_attention.dtype, device=suffix_attention.device)
        attention_mask = torch.cat([prefix_attention, ctx_attention, suffix_attention], dim=1)
        return clip_text_features_from_embeddings(prompt_embeddings, input_ids, attention_mask)

    def all_attribute_features(self, active_ctx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self._features_for_sign("positive", active_ctx), self._features_for_sign("negative", active_ctx)

    def query_direction_from_features(
        self,
        query: str,
        positive_features: torch.Tensor,
        negative_features: torch.Tensor,
        negative_weight: float = 0.5,
    ) -> torch.Tensor:
        pieces = parse_query(query)
        has_pos = any(p.sign == "+" for p in pieces)
        has_neg = any(p.sign == "-" for p in pieces)
        vectors, weights = [], []
        for piece in pieces:
            idx = self.attr_to_idx[piece.raw.lower()]
            if piece.sign == "+":
                vectors.append(positive_features[idx:idx + 1])
                weights.append((1.0 - negative_weight) if has_neg else 1.0)
            else:
                vectors.append(negative_features[idx:idx + 1])
                weights.append(negative_weight if has_pos else 1.0)
        return spherical_weighted_average(vectors, weights)

    def batch_query_directions(self, queries: list[str], active_ctx: int, negative_weight: float = 0.5) -> torch.Tensor:
        positive_features, negative_features = self.all_attribute_features(active_ctx)
        return torch.cat([
            self.query_direction_from_features(query, positive_features, negative_features, negative_weight=negative_weight)
            for query in queries
        ], dim=0)

In [25]:
class ResidualAdapter(nn.Module):
    """Small residual MLP over CLIP projection vectors."""

    def __init__(self, dim: int, max_bottleneck: int = 128):
        super().__init__()
        self.max_bottleneck = int(max_bottleneck)
        self.down = nn.Linear(dim, max_bottleneck, bias=False)
        self.up = nn.Linear(max_bottleneck, dim, bias=False)
        nn.init.normal_(self.down.weight, std=0.02)
        nn.init.zeros_(self.up.weight)

    def forward(self, x: torch.Tensor, active_dim: int) -> torch.Tensor:
        active_dim = int(active_dim)
        if active_dim <= 0:
            return x
        z = F.gelu(self.down(x)[..., :active_dim])
        delta = F.linear(z, self.up.weight[:, :active_dim])
        return normalize(x + delta)


class LowRankResidual(nn.Module):
    """Low-rank residual update over CLIP projection vectors."""

    def __init__(self, dim: int, max_rank: int = 32, scale: float = 1.0):
        super().__init__()
        self.max_rank = int(max_rank)
        self.scale = float(scale)
        self.down = nn.Linear(dim, max_rank, bias=False)
        self.up = nn.Linear(max_rank, dim, bias=False)
        nn.init.normal_(self.down.weight, std=0.02)
        nn.init.zeros_(self.up.weight)

    def forward(self, x: torch.Tensor, rank: int) -> torch.Tensor:
        rank = int(rank)
        if rank <= 0:
            return x
        z = self.down(x)[..., :rank]
        delta = F.linear(z, self.up.weight[:, :rank]) * (self.scale / max(rank, 1))
        return normalize(x + delta)


def apply_module_choice(x, module_choice, adapter, lora, adapter_dim, lora_rank):
    if module_choice in {"adapter", "both"}:
        x = adapter(x, adapter_dim)
    if module_choice in {"lora", "both"}:
        x = lora(x, lora_rank)
    return normalize(x)


class PromptSlerpSearchSupernet(nn.Module):
    """CoOp prompt-token learner plus searchable adapter/LoRA residuals."""

    def __init__(self, attribute_names: list[str], max_ctx: int = 12, max_adapter_dim: int = 128, max_lora_rank: int = 32):
        super().__init__()
        self.prompt_learner = SignedCoOpPromptLearner(attribute_names, max_ctx=max_ctx)
        self.ref_adapter = ResidualAdapter(PROJECTION_DIM, max_adapter_dim)
        self.text_adapter = ResidualAdapter(PROJECTION_DIM, max_adapter_dim)
        self.fused_adapter = ResidualAdapter(PROJECTION_DIM, max_adapter_dim)
        self.ref_lora = LowRankResidual(PROJECTION_DIM, max_lora_rank)
        self.text_lora = LowRankResidual(PROJECTION_DIM, max_lora_rank)
        self.fused_lora = LowRankResidual(PROJECTION_DIM, max_lora_rank)
        self.max_ctx = max_ctx
        self.max_adapter_dim = max_adapter_dim
        self.max_lora_rank = max_lora_rank

    def forward(self, reference_features: torch.Tensor, queries: list[str], candidate: dict) -> torch.Tensor:
        text_features = self.prompt_learner.batch_query_directions(
            queries,
            active_ctx=candidate["active_ctx"],
            negative_weight=candidate["negative_weight"],
        )
        reference_features = normalize(reference_features)
        text_features = normalize(text_features)

        reference_features = apply_module_choice(
            reference_features,
            candidate["reference_module"],
            self.ref_adapter,
            self.ref_lora,
            candidate["adapter_dim"],
            candidate["lora_rank"],
        )
        text_features = apply_module_choice(
            text_features,
            candidate["text_module"],
            self.text_adapter,
            self.text_lora,
            candidate["adapter_dim"],
            candidate["lora_rank"],
        )
        fused = fuse_text_image_slerp(text_features, reference_features, image_alpha=candidate["image_alpha"])
        fused = apply_module_choice(
            fused,
            candidate["fused_module"],
            self.fused_adapter,
            self.fused_lora,
            candidate["adapter_dim"],
            candidate["lora_rank"],
        )
        return normalize(fused)

In [26]:
MODULE_CHOICES = ("none", "adapter", "lora", "both")
FIXED_SEARCH_NEGATIVE_WEIGHT = SELECTED_SLERP_METHOD.negative_weight
FIXED_SEARCH_IMAGE_ALPHA = SELECTED_SLERP_METHOD.image_alpha


def sample_search_candidate(
    active_ctx_choices=(2, 4, 8, 12),
    negative_weights=(FIXED_SEARCH_NEGATIVE_WEIGHT,),
    image_alphas=(FIXED_SEARCH_IMAGE_ALPHA,),
    adapter_dims=(0, 8, 16, 32, 64, 128),
    lora_ranks=(0, 4, 8, 16, 32),
    allow_plain=False,
):
    """Sample one neural prompt-search configuration while keeping SLERP geometry fixed."""
    while True:
        candidate = {
            "active_ctx": int(random.choice(active_ctx_choices)),
            "negative_weight": float(random.choice(negative_weights)),
            "image_alpha": float(random.choice(image_alphas)),
            "adapter_dim": int(random.choice(adapter_dims)),
            "lora_rank": int(random.choice(lora_ranks)),
            "reference_module": random.choice(MODULE_CHOICES),
            "text_module": random.choice(MODULE_CHOICES),
            "fused_module": random.choice(MODULE_CHOICES),
        }
        uses_structure = any(candidate[key] != "none" for key in ["reference_module", "text_module", "fused_module"])
        has_capacity = candidate["adapter_dim"] > 0 or candidate["lora_rank"] > 0
        if allow_plain or (uses_structure and has_capacity):
            return candidate


def conservative_candidate():
    """Deterministic fallback candidate with the selected double-SLERP image-heavy geometry."""
    return {
        "active_ctx": 4,
        "negative_weight": FIXED_SEARCH_NEGATIVE_WEIGHT,
        "image_alpha": FIXED_SEARCH_IMAGE_ALPHA,
        "adapter_dim": 32,
        "lora_rank": 8,
        "reference_module": "none",
        "text_module": "adapter",
        "fused_module": "lora",
    }


pd.DataFrame([sample_search_candidate() for _ in range(5)])

,active_ctx,negative_weight,image_alpha,adapter_dim,lora_rank,reference_module,text_module,fused_module
0,2,0.5,0.8,8,4,none,none,both
1,2,0.5,0.8,8,4,none,adapter,both
2,4,0.5,0.8,0,4,both,lora,lora
3,8,0.5,0.8,128,16,none,both,none
4,8,0.5,0.8,128,0,none,adapter,lora


In [27]:
def build_training_pairs(cases: list[RetrievalCase], positives_per_case: int | None = 1, seed: int = SEED):
    """Expand retrieval cases into (query, reference, positive target) training triples."""
    rng = random.Random(seed)
    pairs = []
    for case in cases:
        positives = list(case.ground_truth)
        if positives_per_case is None or positives_per_case >= len(positives):
            selected = positives
        else:
            selected = rng.sample(positives, positives_per_case)
        for target_id in selected:
            pairs.append({"query": case.query, "reference_id": int(case.reference_id), "target_id": int(target_id)})
    return pairs


def candidate_pool_with_labels(positive_ids: torch.Tensor, image_embeddings: ImageEmbeddings, candidate_pool_size: int | None = None):
    """Map positive image IDs into a candidate-pool row index for cross-entropy training."""
    positive_rows = image_embeddings.rows_for_ids(positive_ids)
    bank_size = image_embeddings.embeddings.shape[0]
    if candidate_pool_size is None or candidate_pool_size >= bank_size:
        candidate_rows = torch.arange(bank_size, device=device)
        labels = positive_rows
    else:
        # The positive rows are always included; random negatives keep the batch affordable when desired.
        random_rows = torch.randint(bank_size, (int(candidate_pool_size),), device=device)
        candidate_rows = torch.cat([positive_rows, random_rows]).unique(sorted=True)
        labels = torch.searchsorted(candidate_rows, positive_rows)
    return candidate_rows, labels.long()


def cuda_amp_dtype() -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def train_prompt_search_supernet(
    supernet: PromptSlerpSearchSupernet,
    train_cases: list[RetrievalCase],
    image_embeddings: ImageEmbeddings,
    epochs: int = 3,
    batch_size: int = 64,
    lr: float = 2e-3,
    candidate_pool_size: int | None = None,
    positives_per_case: int | None = 1,
    fixed_candidate: Optional[dict] = None,
    seed: int = SEED,
    use_amp: bool = True,
    amp_dtype: Optional[torch.dtype] = None,
):
    """Train the supernet with batched GPU scoring against the full or sampled image bank."""
    if not train_cases:
        raise ValueError("train_cases is empty")
    torch.manual_seed(seed)
    random.seed(seed)

    supernet.train()
    optimizer = torch.optim.AdamW(supernet.parameters(), lr=lr, weight_decay=0.0)
    image_bank = image_embeddings.embeddings.to(device)
    logit_scale = model.logit_scale.exp().detach().to(device) if hasattr(model, "logit_scale") else torch.tensor(1 / 0.07, device=device)
    pairs = build_training_pairs(train_cases, positives_per_case=positives_per_case, seed=seed)
    history = []

    use_amp = bool(use_amp and device.type == "cuda")
    amp_dtype = amp_dtype or cuda_amp_dtype()
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp and amp_dtype == torch.float16)

    for epoch in range(1, epochs + 1):
        random.shuffle(pairs)
        losses = []
        progress = tqdm(range(0, len(pairs), batch_size), desc=f"Prompt-search epoch {epoch}/{epochs}")
        for start in progress:
            batch = pairs[start:start + batch_size]
            queries = [item["query"] for item in batch]
            reference_ids = [item["reference_id"] for item in batch]
            positive_ids = torch.tensor([item["target_id"] for item in batch], dtype=torch.long, device=device)

            # fixed_candidate trains one architecture; None trains the one-shot supernet by sampling.
            candidate = fixed_candidate or sample_search_candidate()
            reference_features = image_embeddings.batch_get(reference_ids).to(device)
            candidate_rows, labels = candidate_pool_with_labels(positive_ids, image_embeddings, candidate_pool_size=candidate_pool_size)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp):
                target_features = supernet(reference_features, queries, candidate)
                logits = logit_scale * (target_features @ image_bank[candidate_rows].T)
                loss = F.cross_entropy(logits.float(), labels)

            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            losses.append(float(loss.detach().cpu()))
            progress.set_postfix(loss=np.mean(losses[-10:]), candidates=len(candidate_rows))

        history.append({"epoch": epoch, "loss": float(np.mean(losses)) if losses else float("nan")})
    return pd.DataFrame(history)

In [28]:
@torch.no_grad()
def predict_cases_with_model_batched(
    supernet: PromptSlerpSearchSupernet,
    cases: list[RetrievalCase],
    candidate: dict,
    k: int,
    batch_size: int = EVAL_BATCH_SIZE,
    exclude_reference: bool = EXCLUDE_REFERENCE,
    desc: str = "Prompt-search retrieval",
    show_progress: bool = True,
) -> list[dict[int, float]]:
    """Predict top-k image IDs/scores for many cases with batched CUDA matmul."""
    if not cases:
        return []

    was_training = supernet.training
    supernet.eval()

    image_bank = image_embeddings.embeddings.to(device)
    image_ids = image_embeddings.ids
    predictions_by_case = []
    k_eff = min(int(k), image_bank.shape[0])

    starts = range(0, len(cases), batch_size)
    if show_progress:
        starts = tqdm(starts, desc=desc)

    for start in starts:
        batch_cases = cases[start:start + batch_size]
        reference_ids = [case.reference_id for case in batch_cases]
        queries = [case.query for case in batch_cases]

        reference_features = image_embeddings.batch_get(reference_ids).to(device)
        target_features = supernet(reference_features, queries, candidate)

        # One large [batch, dim] @ [dim, n_images] multiplication keeps CUDA busy.
        scores = target_features @ image_bank.T

        if exclude_reference:
            reference_rows = image_embeddings.rows_for_ids(reference_ids).to(scores.device)
            batch_rows = torch.arange(len(batch_cases), device=scores.device)
            scores[batch_rows, reference_rows] = -float("inf")

        values, rows = torch.topk(scores, k=k_eff, dim=1)
        values = values.detach().cpu()
        rows = rows.detach().cpu()

        for case_values, case_rows in zip(values, rows):
            predictions_by_case.append({
                image_ids[int(row)]: float(value)
                for row, value in zip(case_rows, case_values)
            })

    if was_training:
        supernet.train()

    return predictions_by_case


@torch.no_grad()
def predict_case_with_model(
    supernet: PromptSlerpSearchSupernet,
    case: RetrievalCase,
    candidate: dict,
    k: int,
    exclude_reference: bool = EXCLUDE_REFERENCE,
) -> dict[int, float]:
    """Single-case wrapper around the batched neural retrieval path."""
    return predict_cases_with_model_batched(
        supernet,
        [case],
        candidate,
        k=k,
        batch_size=1,
        exclude_reference=exclude_reference,
        show_progress=False,
    )[0]


def evaluate_cases_with_model(
    supernet: PromptSlerpSearchSupernet,
    cases: list[RetrievalCase],
    candidate: dict,
    ks: Sequence[int] = KS,
    method_name: str = "selected_prompt_slerp_search",
    batch_size: int = EVAL_BATCH_SIZE,
    show_progress: bool = True,
) -> pd.DataFrame:
    """Evaluate one selected model/candidate with batched GPU prediction."""
    predictions_by_case = predict_cases_with_model_batched(
        supernet,
        cases,
        candidate,
        k=max(ks),
        batch_size=batch_size,
        exclude_reference=EXCLUDE_REFERENCE,
        desc=method_name,
        show_progress=show_progress,
    )

    rows = []
    iterator = zip(cases, predictions_by_case)
    if show_progress:
        iterator = tqdm(list(iterator), desc=f"Scoring {method_name}")

    for case, predictions in iterator:
        retrieved = list(predictions.keys())
        row = {
            "method": method_name,
            "query_id": case.query_id,
            "query": case.query,
            "reference_id": case.reference_id,
            "n_gt": len(case.ground_truth),
            "batch_size": batch_size,
            "exclude_reference": EXCLUDE_REFERENCE,
        }
        row.update(candidate)
        for k in ks:
            row.update(evaluate_retrieval(retrieved, case.ground_truth, k))
        rows.append(row)
    return pd.DataFrame(rows)


def search_candidates(
    supernet: PromptSlerpSearchSupernet,
    validation_cases: list[RetrievalCase],
    num_candidates: int = 24,
    ks: Sequence[int] = KS,
    score_metric: str = "Recall@10",
    seed: int = SEED,
    eval_batch_size: int = EVAL_BATCH_SIZE,
):
    """Evaluate sampled candidates and return the best configuration by a chosen metric."""
    if not validation_cases:
        raise ValueError("validation_cases is empty")
    rng_state = random.getstate()
    random.seed(seed)
    rows = []
    best_candidate = None
    best_score = -float("inf")

    for candidate_id in tqdm(range(num_candidates), desc="Searching prompt candidates"):
        candidate = sample_search_candidate(allow_plain=True)
        df = evaluate_cases_with_model(
            supernet,
            validation_cases,
            candidate,
            ks=ks,
            method_name=f"candidate_{candidate_id}",
            batch_size=eval_batch_size,
            show_progress=False,
        )
        score = float(df[score_metric].mean())
        row = {"candidate_id": candidate_id, "score_metric": score_metric, "score": score, **candidate}
        rows.append(row)
        if score > best_score:
            best_score = score
            best_candidate = candidate

    random.setstate(rng_state)
    return pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True), best_candidate

In [29]:
# Full prompt-search configuration. These defaults are meant for the same 14/15 GB VRAM setup:
# evaluation uses large batches; training can use the full image bank unless memory is tight.
PROMPT_TRAIN_FRACTION = 0.70
PROMPT_VALIDATION_FRACTION = 0.15
PROMPT_TEST_FRACTION = 1.0 - PROMPT_TRAIN_FRACTION - PROMPT_VALIDATION_FRACTION
PROMPT_EPOCHS = 3
PROMPT_TRAIN_BATCH_SIZE = 64
PROMPT_EVAL_BATCH_SIZE = EVAL_BATCH_SIZE
PROMPT_SEARCH_CANDIDATES = 24
PROMPT_LR = 2e-3
PROMPT_POSITIVES_PER_CASE = 1
PROMPT_CANDIDATE_POOL_SIZE = None  # None = score against the full image bank during training.

prompt_train_cases, prompt_validation_cases, prompt_test_cases = split_cases_by_query(
    FULL_CASES,
    train_fraction=PROMPT_TRAIN_FRACTION,
    validation_fraction=PROMPT_VALIDATION_FRACTION,
    seed=SEED,
)

if not prompt_validation_cases:
    prompt_validation_cases = prompt_train_cases
if not prompt_test_cases:
    prompt_test_cases = prompt_validation_cases

print(f"Prompt-search train cases: {len(prompt_train_cases)}")
print(f"Prompt-search validation cases: {len(prompt_validation_cases)}")
print(f"Prompt-search held-out test cases: {len(prompt_test_cases)}")

prompt_split_summary_df = pd.concat(
    [
        pd.DataFrame({"split": "train", "query_id": [c.query_id for c in prompt_train_cases], "query": [c.query for c in prompt_train_cases]}),
        pd.DataFrame({"split": "validation", "query_id": [c.query_id for c in prompt_validation_cases], "query": [c.query for c in prompt_validation_cases]}),
        pd.DataFrame({"split": "test", "query_id": [c.query_id for c in prompt_test_cases], "query": [c.query for c in prompt_test_cases]}),
    ],
    ignore_index=True,
)
prompt_split_summary_df = prompt_split_summary_df.groupby(["split", "query_id", "query"], as_index=False).size().rename(columns={"size": "n_cases"})

prompt_supernet = PromptSlerpSearchSupernet(
    raw_attributes,
    max_ctx=12,
    max_adapter_dim=128,
    max_lora_rank=32,
).to(device)

prompt_history_df = train_prompt_search_supernet(
    prompt_supernet,
    prompt_train_cases,
    image_embeddings,
    epochs=PROMPT_EPOCHS,
    batch_size=PROMPT_TRAIN_BATCH_SIZE,
    lr=PROMPT_LR,
    candidate_pool_size=PROMPT_CANDIDATE_POOL_SIZE,
    positives_per_case=PROMPT_POSITIVES_PER_CASE,
    fixed_candidate=None,
    seed=SEED,
)

prompt_history_df

Prompt-search train cases: 23138
Prompt-search validation cases: 4957
Prompt-search held-out test cases: 4957


/tmp/ipykernel_37915/1753763165.py:66: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp and amp_dtype == torch.float16)


Prompt-search epoch 1/3:   0%|          | 0/362 [00:00<?, ?it/s]

Prompt-search epoch 2/3:   0%|          | 0/362 [00:00<?, ?it/s]

Prompt-search epoch 3/3:   0%|          | 0/362 [00:00<?, ?it/s]

,epoch,loss
0,1,15.418667
1,2,12.307304
2,3,11.429976


In [30]:
prompt_search_table, prompt_best_candidate = search_candidates(
    prompt_supernet,
    prompt_validation_cases,
    num_candidates=PROMPT_SEARCH_CANDIDATES,
    ks=KS,
    score_metric="Recall@10",
    seed=SEED,
    eval_batch_size=PROMPT_EVAL_BATCH_SIZE,
)

print("Best prompt-search candidate:")
print(prompt_best_candidate)
prompt_search_table

Searching prompt candidates:   0%|          | 0/24 [00:00<?, ?it/s]

Best prompt-search candidate:
{'active_ctx': 4, 'negative_weight': 0.5, 'image_alpha': 0.8, 'adapter_dim': 16, 'lora_rank': 0, 'reference_module': 'adapter', 'text_module': 'adapter', 'fused_module': 'none'}


,candidate_id,score_metric,score,active_ctx,negative_weight,image_alpha,adapter_dim,lora_rank,reference_module,text_module,fused_module
0,20,Recall@10,0.255195,4,0.5,0.8,16,0,adapter,adapter,none
1,7,Recall@10,0.248336,12,0.5,0.8,16,8,adapter,lora,none
2,19,Recall@10,0.228162,4,0.5,0.8,128,8,both,both,none
3,0,Recall@10,0.226750,2,0.5,0.8,8,4,adapter,none,none
4,15,Recall@10,0.217672,4,0.5,0.8,128,32,adapter,adapter,lora
5,10,Recall@10,0.207989,8,0.5,0.8,32,16,adapter,lora,adapter
6,5,Recall@10,0.206173,12,0.5,0.8,128,32,lora,adapter,none
7,2,Recall@10,0.201331,12,0.5,0.8,64,8,none,adapter,both
8,22,Recall@10,0.197498,8,0.5,0.8,64,4,both,adapter,both
9,11,Recall@10,0.195683,4,0.5,0.8,64,16,lora,adapter,adapter


In [31]:
neural_prompt_results_df = evaluate_cases_with_model(
    prompt_supernet,
    prompt_test_cases,
    prompt_best_candidate,
    ks=KS,
    method_name="neural_prompt_search_heldout_test_double_slerp_image_heavy",
    batch_size=PROMPT_EVAL_BATCH_SIZE,
)

neural_prompt_query_table = summarize_results_by_query(neural_prompt_results_df)
neural_prompt_overall_table = summarize_results_overall(neural_prompt_results_df)
neural_prompt_query_table

neural_prompt_search_heldout_test_double_slerp_image_heavy:   0%|          | 0/10 [00:00<?, ?it/s]

Scoring neural_prompt_search_heldout_test_double_slerp_image_heavy:   0%|          | 0/4957 [00:00<?, ?it/s]

,method,query_id,query,n_cases,Recall@1,Precision@1,Recall@5,Precision@5,Recall@10,Precision@10
0,neural_prompt_search_heldout_test_double_slerp...,0,+Smiling,718,0.061281,0.061281,0.155989,0.039554,0.238162,0.033287
1,neural_prompt_search_heldout_test_double_slerp...,1,+Eyeglasses,329,0.203647,0.203647,0.471125,0.165350,0.595745,0.139210
2,neural_prompt_search_heldout_test_double_slerp...,2,-Heavy_Makeup,613,0.032626,0.032626,0.104405,0.022512,0.187602,0.022186
3,neural_prompt_search_heldout_test_double_slerp...,3,+Male,239,0.058577,0.058577,0.196653,0.055230,0.297071,0.049372
4,neural_prompt_search_heldout_test_double_slerp...,4,-Young,803,0.018680,0.018680,0.098381,0.025903,0.166874,0.024533
5,neural_prompt_search_heldout_test_double_slerp...,5,+Blond_Hair,820,0.063415,0.063415,0.168293,0.042439,0.269512,0.041341
6,neural_prompt_search_heldout_test_double_slerp...,6,+Mustache,45,0.022222,0.022222,0.177778,0.035556,0.311111,0.035556
7,neural_prompt_search_heldout_test_double_slerp...,7,-Young,803,0.034869,0.034869,0.129514,0.033126,0.201743,0.030760
8,neural_prompt_search_heldout_test_double_slerp...,8,"+Eyeglasses, +Smiling",92,0.173913,0.173913,0.413043,0.110870,0.630435,0.095652
9,neural_prompt_search_heldout_test_double_slerp...,9,"+Black_Hair, -Wavy_Hair",386,0.054404,0.054404,0.150259,0.032642,0.238342,0.031088


In [34]:
# # Qualitative neural prompt-search example. Uncomment only when you want images/IDs in the notebook output.
# neural_case = prompt_test_cases[0]
# neural_predictions = predict_case_with_model(
#     prompt_supernet,
#     neural_case,
#     prompt_best_candidate,
#     k=5,
# )
# plot_case_predictions(
#     neural_case,
#     neural_predictions,
#     k_to_plot=5,
#     title="Neural prompt search selected candidate on held-out test",
# )

---
## Compact Overall Tables
---

The main required outputs above are the per-query quantitative tables. These compact summaries are included only to make method ranking easier after the full run completes.

In [35]:
all_overall_tables = pd.concat(
    [
        baseline_overall_table.assign(section="baseline"),
        simple_slerp_overall_table.assign(section="simple_slerp"),
        multiple_slerp_overall_table.assign(section="multiple_slerp"),
        selected_slerp_overall_table.assign(section="selected_slerp"),
        neural_prompt_overall_table.assign(section="neural_prompt_search"),
    ],
    ignore_index=True,
)

front_cols = ["section", "method", "n_cases"]
metric_cols = [c for c in all_overall_tables.columns if c.startswith(("Recall@", "Precision@"))]
all_overall_tables[front_cols + metric_cols].sort_values(["Recall@10", "Precision@10"], ascending=False)

,section,method,n_cases,Recall@1,Precision@1,Recall@5,Precision@5,Recall@10,Precision@10
20,neural_prompt_search,neural_prompt_search_heldout_test_double_slerp...,4957,0.056082,0.056082,0.163809,0.044019,0.254388,0.040145
14,multiple_slerp,"12 Text SLERP + SLERP image fusion, negative-h...",33052,0.018395,0.018395,0.060148,0.015049,0.092642,0.013324
17,multiple_slerp,"15 Text SLERP + SLERP image fusion, text-light...",33052,0.018395,0.018395,0.060117,0.015049,0.092612,0.013321
11,multiple_slerp,"09 Text SLERP + SLERP image fusion, balanced/i...",33052,0.018395,0.018395,0.060027,0.015031,0.092612,0.013318
19,selected_slerp,"09 Text SLERP + SLERP image fusion, balanced/i...",33052,0.018395,0.018395,0.060027,0.015031,0.092612,0.013318
2,simple_slerp,"03 SLERP image fusion, image-heavy",33052,0.018637,0.018637,0.060722,0.015134,0.091492,0.013209
5,multiple_slerp,"03 SLERP image fusion, image-heavy",33052,0.018637,0.018637,0.060722,0.015134,0.091492,0.013209
1,baseline,01 Arithmetic text + image,33052,0.018335,0.018335,0.059331,0.014692,0.091159,0.012865
3,multiple_slerp,01 Arithmetic text + image,33052,0.018335,0.018335,0.059331,0.014692,0.091159,0.012865
4,multiple_slerp,"02 SLERP image fusion, balanced",33052,0.018335,0.018335,0.059331,0.014692,0.091159,0.012865
